# NB05bV5 — Per-Point Kernel Dataset Builder

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


V5 mirrors V4's parquet structure (28 atomic + 8 fusion) and keeps V4's
point-based sample inventory exactly. The only architectural change vs V4 is
the kernel size: KERNEL_SIZE = 5 (5x5 = 25 pixels per point) instead of V4's
3x3 = 9. No SAR speckle filter.

| | V2 (footprint) | V3 (1 pixel) | V4 (3x3 kernel) | V5 (5x5 kernel) |
|---|---|---|---|---|
| spatial extent per row | variable (1-N pixels) | 1 pixel | always 9 pixels | always 25 pixels |
| reductions per modality | v29 8-stat | 1 sample | v29 8-stat over 9 px | v29 8-stat over 25 px |
| SAR speckle handling | footprint multi-look | exposed | small 3x3 multi-look | medium 5x5 multi-look |

V5 sits one step up the spatial-extent gradient from V4. Used to test
whether the V4 kernel size was insufficient for the matched-filter benefit
to materialize on SAR/wide-accumulator parquets.

Output: `STACK_DIR/dataset/v5/`. Helpers: nb05bv5_helpers.py.

Same point inventory as V3/V4 (UNOSAT positives + sampled centroids).
Same parquet inventory, same NB03e v48 source TIFs, same metadata
taxonomy, same tier system, same GroupKFold cell.

In [1]:
# @title CELL 1: NB05bV5 CONFIG
TIER_SELECTION = [0,1,2]
CITY_SELECTION = None
REQUIRE_UNOSAT = True

# negative sampling (V3-specific)
NEG_RATIO = 5
NEG_MIN_DIST_M = 30.0
NEG_SEED = 42
DROP_EXCLUDED = True   # drop UNOSAT damage_binary == -1 (Impact Crater)

# V5 kernel-based zonal aggregation (override applied to nb05bv5_helpers in CELL 3)
KERNEL_SIZE = 5   # 3x3 = 9 pixels per point, NaN-padded at AOI edges (v29 small-N rule applies)

# per-parquet force rerun (False = skip if parquet exists)
FR_POINTS = True
FR_SCENE_MS = True
FR_SCENE_CARD = True
FR_SCENE_COH = True
FR_SCENE_LANDUSE = True
FR_SCENE_INDICES = True
FR_SCENE_NBR = True
FR_ROLLING_COH = True
FR_ROLLING_CARD = True
FR_COMPOSITE_PREPOST_BANDS = True
FR_COMPOSITE_PREPOST_LANDUSE = True
FR_COMPOSITE_VS_SCENES_BANDS = True
FR_COMPOSITE_VS_SCENES_LANDUSE = True
FR_PREPOST_SINGLE_CARD = True
FR_COH_DROP = True
FR_CARD_DROP = True
FR_MS_CHANGE = True
FR_MS_MAHA = True
FR_LU_CHANGE = True
FR_BLOCK_STATS = True
FR_ROLLING_STATS = True
FR_ROLLING_ACCUM_COH = True
FR_ROLLING_ACCUM_CARD = True
FR_ROLLING_ACCUM_MS = True
FR_BLOCK_ACCUM_COH = True
FR_BLOCK_ACCUM_CARD = True
FR_BLOCK_ACCUM_MS = True
FR_FUSIONS = True
FR_GROUPKFOLD = True
FR_MANIFEST = True


In [2]:
# @title CELL 2: LOAD GLOBAL SETUP
exec(open(NOTEBOOKS_DIR / 'global_setup.py').read()) if 'NOTEBOOKS_DIR' in dir() \
    else exec(open('/content/drive_f/masterthesis/notebooks/global_setup.py').read())
print(f"  STACK_ROOT: {STACK_ROOT}")
print(f"  STACK_DIR (local primary): {STACK_DIR}")
print(f"  NOTEBOOKS_DIR on sys.path: {NOTEBOOKS_DIR}")


BDA GLOBAL SETUP
Started: 2026-04-27 23:53:00
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     909.8/7452.0 GB (6542.2 GB free)
  GDrive (F:)     1392.3/3726.0 GB (2333.7 GB free)
  Local data      11557.6/14901.9 GB (3344.3 GB free)
  Data stack      1392.3/3726.0 GB (2333.7 GB free)
  WSL ext4        68.6/1006.9 GB (887.1 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

# SHARED: Load V3 helpers + discover cities (grouped by tier)

In [3]:
# @title CELL 3: V5 SHARED HELPERS + CITY DISCOVERY (PER-TIER)
import sys, importlib, re, time, gc, json
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ---- v4 output directory ----
DATASET_ROOT_V5 = STACK_DIR / 'dataset' / 'v5'
DATASET_ROOT_V5.mkdir(parents=True, exist_ok=True)
V5_DIR = DATASET_ROOT_V5
print(f"  DATASET_ROOT_V5: {DATASET_ROOT_V5}")

def v5_path(name, tier):
    return V5_DIR / f"bda_{name}_t{tier}.parquet"

# ---- catalog (same DB as V2) ----
from stack_catalog import BDACatalog
cat = BDACatalog(CATALOG_DB)
print(f"  Catalog: {CATALOG_DB}")

# ---- V5 helpers (kernel-based sampling; reuses V3 data loaders) ----
import nb05bv5_helpers
importlib.reload(nb05bv5_helpers)
nb05bv5_helpers.KERNEL_SIZE = KERNEL_SIZE   # apply CONFIG override to module-level default
from nb05bv5_helpers import (
    sample_one_kernel,
    load_unosat_points_for_city, load_building_centroids,
    build_negative_samples_from_buildings, precompute_point_indices,
    sample_raster_block,
)
nb05bv5_helpers._print_banner()

UNOSAT_ROOT = UNOSAT_CITIES_DIR
print(f"  UNOSAT_ROOT: {UNOSAT_ROOT}")

# ---- discover ML-ready cities ----
manifest_path = STACK_ROOT / "data_stack_manifest.json"
with open(manifest_path) as f:
    manifest = json.load(f)

city_meta = {}
for city_name, info in sorted(manifest.get("cities", {}).items()):
    tier = info.get("tier", 99)
    if TIER_SELECTION and tier not in TIER_SELECTION:
        continue
    if CITY_SELECTION and city_name not in (CITY_SELECTION if isinstance(CITY_SELECTION, list) else [CITY_SELECTION]):
        continue
    if not info.get("ready_ml", False):
        continue
    ref_path = STACK_ROOT / city_name / "reference_grid.json"
    if not ref_path.exists():
        continue
    if REQUIRE_UNOSAT:
        unosat_path = UNOSAT_ROOT / city_name / "unosat_damage.geojson"
        if not unosat_path.exists():
            continue
    aoi_row = load_aoi(city_name)
    city_meta[city_name] = {
        "battle_start": str(aoi_row.get("battle_start", ""))[:10],
        "battle_stop": str(aoi_row.get("battle_stop", "") or "")[:10],
        "tier": int(aoi_row.get("tier", 99)),
        "manifest": info,
    }

CITIES = sorted(city_meta.keys())
TIER_CITIES = defaultdict(list)
for c in CITIES:
    TIER_CITIES[city_meta[c]['tier']].append(c)
TIER_CITIES = dict(sorted(TIER_CITIES.items()))

print(f"\nCities: {len(CITIES)} across {len(TIER_CITIES)} tiers")
for tier, tc in TIER_CITIES.items():
    print(f"  Tier {tier}: {len(tc)} cities")
    for c in tc:
        m = city_meta[c]
        print(f"    {c:<22s} battle={m['battle_start']}")


# ============================================================================
# V3 CORE: per-city point table builder (cached)
# ============================================================================
_point_table_cache = {}

def build_or_load_points(city_name):
    """Build the per-city point table (UNOSAT positives + UNOSAT undamaged
    + sampled building-centroid negatives) with row/col precomputed.

    Returns:
      (pts_df, ref_shape, transform)
      where pts_df has columns: point_id, unosat_id, building_id, city,
      point_source, damage, damage_label, damage_binary, ep, t_unosat,
      lon, lat, x_utm, y_utm, row, col.
    """
    if city_name in _point_table_cache:
        return _point_table_cache[city_name]

    city_stack = STACK_ROOT / city_name
    rg_path = city_stack / "reference_grid.json"
    if not rg_path.exists():
        return None, None, None
    rg = json.load(open(rg_path))
    transform = rasterio.transform.Affine(*rg['transform'][:6])
    ref_shape = (rg['height'], rg['width'])
    raster_crs = rg['crs']

    # 1. UNOSAT points (positives + undamaged; excluded dropped per DROP_EXCLUDED)
    gdf_unosat = load_unosat_points_for_city(
        city_name, raster_crs, UNOSAT_ROOT,
        drop_excluded=DROP_EXCLUDED,
    )

    # 2. Building centroids (vectorized via np.bincount on building_labels.tif)
    centroids = load_building_centroids(city_name, STACK_ROOT)

    # 3. Sample N=NEG_RATIO*n_positives building-centroid negatives, >=NEG_MIN_DIST_M from positives
    positives = gdf_unosat[gdf_unosat['damage_binary'] == 1].reset_index(drop=True)
    if len(positives) > 0 and len(centroids) > 0:
        negatives, _ = build_negative_samples_from_buildings(
            positive_points=positives,
            building_centroids=centroids,
            raster_crs=raster_crs,
            city=city_name,
            ratio=NEG_RATIO,
            min_dist_m=NEG_MIN_DIST_M,
            seed=NEG_SEED,
        )
    else:
        negatives = gdf_unosat.iloc[0:0].copy()

    # 4. Combine + precompute pixel indices
    unified = pd.concat([gdf_unosat, negatives], ignore_index=True)
    unified = precompute_point_indices(unified, transform, ref_shape)

    # 5. Encode UNOSAT date as int32 YYYYMMDD (-1 if missing)
    def _to_yyyymmdd(d):
        if pd.isna(d):
            return -1
        try:
            return int(pd.to_datetime(d).strftime('%Y%m%d'))
        except (ValueError, TypeError):
            return -1
    unified['t_unosat'] = unified['date'].map(_to_yyyymmdd).astype(np.int32)

    # 6. point_id stable across notebook runs
    unified['point_id'] = (
        unified['city'].astype(str) + '_'
        + unified['point_source'].astype(str) + '_'
        + unified.index.astype(str)
    )

    # 7. Cross-link to building footprint (optional V2 cross-reference)
    labels_path = city_stack / "building_labels.tif"
    if labels_path.exists():
        with rasterio.open(labels_path) as src:
            labels_arr = src.read(1)
        H_l, W_l = labels_arr.shape
        rows_pt = unified['row'].values
        cols_pt = unified['col'].values
        in_bounds = ((rows_pt >= 0) & (rows_pt < H_l)
                     & (cols_pt >= 0) & (cols_pt < W_l))
        bldg_id_at_pt = np.full(len(unified), -1, dtype=np.int32)
        if in_bounds.any():
            bldg_id_at_pt[in_bounds] = labels_arr[rows_pt[in_bounds], cols_pt[in_bounds]]
        if 'building_id' in unified.columns:
            existing = unified['building_id'].values
            unified['building_id'] = np.where(
                pd.isna(existing) | (existing == 0) | (existing == -1),
                bldg_id_at_pt, existing,
            ).astype(np.int32)
        else:
            unified['building_id'] = bldg_id_at_pt
    else:
        unified['building_id'] = -1

    # Drop the raw UNOSAT date column (we have t_unosat) and geometry
    drop_cols = [c for c in ['date', 'geometry'] if c in unified.columns]
    if drop_cols:
        unified = unified.drop(columns=drop_cols)

    _point_table_cache[city_name] = (unified, ref_shape, transform)
    return _point_table_cache[city_name]


# ============================================================================
# V3 CORE: sample one TIF at all points of a city
# ============================================================================
def sample_one(tif_path, pts_df, ref_shape):
    """Sample one TIF at each point's pixel (row, col).

    Returns 1-D float32 array of length len(pts_df). Out-of-bounds and nodata
    pixels become NaN.
    """
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            try:
                nd_f = float(nodata)
                if not np.isnan(nd_f):
                    data[data == nd_f] = np.nan
            except (ValueError, TypeError):
                pass
        if data.shape != ref_shape:
            H, W = ref_shape
            padded = np.full(ref_shape, np.nan, dtype=np.float32)
            h = min(data.shape[0], H); w = min(data.shape[1], W)
            padded[:h, :w] = data[:h, :w]
            data = padded
    return sample_raster_block(data, pts_df)


# ============================================================================
# Period / timestep helpers (unchanged from v29)
# ============================================================================
def get_period_label(date_str, battle_start, battle_stop):
    d_clean = str(date_str).replace('-', '')[:8]
    dt = datetime.strptime(d_clean, '%Y%m%d')
    bs = datetime.strptime(str(battle_start)[:10], '%Y-%m-%d')
    if dt < bs:
        return 'prebattle'
    be = None
    if battle_stop and str(battle_stop).lower() not in ('', 'none', 'nat', 'ongoing'):
        be = datetime.strptime(str(battle_stop)[:10], '%Y-%m-%d')
    if be and dt > be:
        return 'postbattle'
    return 'crossbattle'

def assign_timesteps(dates, battle_start_str):
    bs = datetime.strptime(battle_start_str, "%Y-%m-%d")
    dated = [(d, datetime.strptime(d, "%Y%m%d")) for d in dates]
    pre = sorted([(d, dt) for d, dt in dated if dt < bs], key=lambda x: x[1])
    post = sorted([(d, dt) for d, dt in dated if dt >= bs], key=lambda x: x[1])
    ts = {}
    for i, (d, dt) in enumerate(pre):
        ts[d] = -(len(pre) - i)
    for i, (d, dt) in enumerate(post):
        ts[d] = i
    return ts


# ============================================================================
# V4: column-name builder for kernel-aware sampling
# ============================================================================
def _v4_col(base, stat):
    """Return the V4 column name for a (base, stat) pair.

    For numeric stats (mean, p10, p50, p90, std, min, max, max_abs_delta):
        base + "__" + stat
    For the categorical 'mode' stat:
        base   (no suffix; matches V3 naming for categorical TIFs)
    """
    if stat == 'mode':
        return base
    return f"{base}__{stat}"


# ============================================================================
# COLUMN ROLE DEFINITIONS — V5 (point_id, kernel-aggregated features)
# ============================================================================
ID_COLS = {'point_id'}
LABEL_COLS = {'damage_binary', 'damage_label', 'damage', 'ems98_grade'}
META_COLS_SET = {
    'city', 'tier', 'battle_start', 'battle_stop', 'conflict_ongoing',
    'has_card', 'has_coh', 'has_ms',
    # V3 point identity / linkage
    'point_source', 'unosat_id', 'building_id', 'ep', 't_unosat',
    'lon', 'lat', 'x_utm', 'y_utm', 'row', 'col',
    # legacy V2 metadata that still appears in some derived columns
    'unosat_date', 'unosat_ep', 'match_method', 'match_distance', 'in_aoi',
    'height', 'num_floors', 'roof_height',
    'area_m2', 'centroid_x', 'centroid_y', 'n_pixels',
    'date', 'date1', 'date2', 'timestep', 'period_label',
    'was_observed_ms', 'was_observed_card', 'was_observed_coh',
    'was_observed_cohdrop', 'was_observed_blockstats',
    'pre_date_card', 'post_date_card', 'dataset',
}

def _is_metadata_column(col):
    """V5: point_id is the ID; building_id is metadata (cross-reference to V2)."""
    if col in META_COLS_SET:
        return True
    if col in ID_COLS or col in LABEL_COLS:
        return True
    if col.startswith('was_observed_'):
        return True
    return False

def build_role_overrides(columns):
    overrides = {}
    for col in columns:
        if col in ID_COLS:
            overrides[col] = 'id'
        elif col in LABEL_COLS:
            overrides[col] = 'label'
        elif _is_metadata_column(col):
            overrides[col] = 'metadata'
    return overrides

print("  Roles: ID_COLS, LABEL_COLS, META_COLS_SET, build_role_overrides()")


# ============================================================================
# Profile logger (V3-native, writes to V5_DIR/dataset_profiles)
# ============================================================================
PROFILE_DIR = V5_DIR / 'dataset_profiles'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

def log_dataset_profile(df, parquet_name):
    from datetime import datetime as _dt
    num_cols = [c for c in df.columns if df[c].dtype.kind in ('f', 'i', 'u')]
    feat_cols = [c for c in num_cols if not _is_metadata_column(c)]

    profile = {
        'parquet_name': parquet_name,
        'timestamp': _dt.now().isoformat(),
        'n_rows': int(len(df)),
        'n_cols': int(len(df.columns)),
        'n_features': len(feat_cols),
        'n_cities': int(df['city'].nunique()) if 'city' in df.columns else 0,
        'nan_rate_overall': float(df[feat_cols].isna().mean().mean()) if feat_cols else 0.0,
    }

    if 'city' in df.columns:
        profile['points_per_city'] = {
            c: int(n) for c, n in df['city'].value_counts().items()
        }

    if 'city' in df.columns and 'damage_binary' in df.columns:
        dr = df.groupby('city')['damage_binary'].mean()
        profile['damage_rate_per_city'] = {c: float(v) for c, v in dr.items()}

    if 'city' in df.columns and feat_cols:
        nan_per_city = df.groupby('city')[feat_cols].apply(
            lambda g: g.isna().mean().mean()
        )
        profile['nan_rate_per_city'] = {c: float(v) for c, v in nan_per_city.items()}

    out = PROFILE_DIR / f'{parquet_name}.json'
    with open(out, 'w') as f:
        json.dump(profile, f, indent=2)

    from scipy import stats as _stats
    feat_records = []
    for col in feat_cols:
        s = df[col]
        n_nan = int(s.isna().sum())
        nan_pct = 100.0 * n_nan / len(s) if len(s) > 0 else 0.0
        valid = s.dropna()
        rec = {'feature': col, 'nan_pct': round(nan_pct, 2)}
        if len(valid) > 1:
            m = float(valid.mean())
            sd = float(valid.std())
            rec['mean'] = round(m, 6)
            rec['std'] = round(sd, 6)
            rec['min'] = round(float(valid.min()), 6)
            rec['max'] = round(float(valid.max()), 6)
            rec['cv'] = round(sd / abs(m), 4) if abs(m) > 1e-12 else 0.0
            rec['skewness'] = round(float(_stats.skew(valid, nan_policy='omit')), 4)
            rec['kurtosis'] = round(float(_stats.kurtosis(valid, nan_policy='omit')), 4)
        else:
            rec.update({'mean': None, 'std': None, 'min': None, 'max': None,
                        'cv': None, 'skewness': None, 'kurtosis': None})
        feat_records.append(rec)

    feat_out = PROFILE_DIR / f'{parquet_name}_features.json'
    with open(feat_out, 'w') as f:
        json.dump(feat_records, f, indent=1)
    print(f"    profile -> {out.name} + {feat_out.name} ({len(feat_records)} features)")


# ============================================================================
# Wide-parquet imputation (unchanged from v29)
# ============================================================================
def impute_wide_parquet(df, modality_name):
    feat_cols = [c for c in df.columns
                 if not _is_metadata_column(c) and df[c].dtype.kind in ('f', 'i', 'u')]
    if not feat_cols:
        return df
    flag_col = f"was_observed_{modality_name}"
    df[flag_col] = df[feat_cols].notna().any(axis=1).astype(int)
    medians = df[feat_cols].median()
    n_before = df[feat_cols].isna().sum().sum()
    df[feat_cols] = df[feat_cols].fillna(medians)
    n_after = df[feat_cols].isna().sum().sum()
    print(f"    impute_wide_parquet({modality_name}): {n_before} NaN -> {n_after} NaN, {flag_col} added")
    return df


# ============================================================================
# Save + register
# ============================================================================
def save_v5_parquet(df, name, tier):
    out_path = v5_path(name, tier)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, index=False)
    pq_name = f"bda_{name}_t{tier}"
    cat.register_parquet_columns(pq_name, df.columns,
                                  role_overrides=build_role_overrides(df.columns))
    log_dataset_profile(df, pq_name)
    return out_path

MANIFEST_ENTRIES = {}

def register_manifest(name, pq_id, fmt, join_keys, feature_columns, question, tier, n_rows,
                       depends_on_tifs=None, composed_of=None, cities_excluded=None):
    _prev_excl = MANIFEST_ENTRIES.get(name, {}).get('cities_excluded_per_tier', {})
    MANIFEST_ENTRIES[name] = {
        'id': pq_id,
        'pattern': f"v5/bda_{name}_t{{tier}}.parquet",
        'format': fmt,
        'join_keys': join_keys,
        'feature_columns': feature_columns,
        'n_features': len(feature_columns),
        'experiment_question': question,
        'tiers_built': MANIFEST_ENTRIES.get(name, {}).get('tiers_built', []) + [tier],
        'n_rows_per_tier': {**MANIFEST_ENTRIES.get(name, {}).get('n_rows_per_tier', {}), str(tier): n_rows},
    }
    if depends_on_tifs:
        MANIFEST_ENTRIES[name]['depends_on_tifs'] = depends_on_tifs
    if composed_of:
        MANIFEST_ENTRIES[name]['composed_of'] = composed_of
    if cities_excluded is not None:
        MANIFEST_ENTRIES[name]['cities_excluded_per_tier'] = {**_prev_excl, str(tier): sorted(list(cities_excluded))}
    elif _prev_excl:
        MANIFEST_ENTRIES[name]['cities_excluded_per_tier'] = _prev_excl

print("  Manifest accumulator ready")


# ============================================================================
# PARQUET CATALOG (V3) + scan_disk_and_register
# ============================================================================
PARQUET_CATALOG = {
    'points':                       {'id': 'A0',  'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'V3 point identity table'},
    'scene_ms':                     {'id': 'A1',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q1: Does optical carry per-scene damage signal?'},
    'scene_card':                   {'id': 'A2',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q1: Does CARD carry per-scene damage signal?'},
    'scene_coh':                    {'id': 'A3',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q1: Does COH carry per-scene damage signal?'},
    'scene_landuse':                {'id': 'A4',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q5: Does landuse context help?'},
    'scene_indices':                {'id': 'A5',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q1: Do spectral indices outperform raw bands?'},
    'scene_nbr':                    {'id': 'A6',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q1: Does NBR carry signal?'},
    'rolling_coh':                  {'id': 'A7',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q4: Do rolling windows add value (COH)?'},
    'rolling_card':                 {'id': 'A8',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q4: Do rolling windows add value (CARD)?'},
    'composite_prepost_bands':      {'id': 'A9',  'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q3: Do composites work? (Dietrich baseline)'},
    'composite_prepost_landuse':    {'id': 'A10', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q5: Does composite landuse change help?'},
    'composite_vs_scenes_bands':    {'id': 'A11', 'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q3: Does per-scene trajectory beat static snapshot?'},
    'composite_vs_scenes_landuse':  {'id': 'A12', 'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q5: Per-scene landuse trajectory?'},
    'prepost_single_card':          {'id': 'A13', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q3: Single-scene CARD pre/post'},
    'coh_drop':                     {'id': 'A14', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q1: Does cumulative COH drop carry signal?'},
    'block_stats':                  {'id': 'A15', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q6: Does Dietrich block approach work with GroupKFold?'},
    'rolling_stats_roll3':          {'id': 'A16', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q4: Rolling stats (window=3)'},
    'rolling_stats_roll7':          {'id': 'A17', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q4: Rolling stats (window=7)'},
    'rolling_stats_roll13':         {'id': 'A18', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q4: Rolling stats (window=13)'},
    'card_drop':                    {'id': 'A19', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q: Does cumulative CARD z-score drop carry signal?'},
    'ms_change':                    {'id': 'A20', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q: Does cumulative MS SWIR brightness + NBR anomaly carry signal?'},
    'ms_maha':                      {'id': 'A21', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q: Does multi-band MS Mahalanobis distance carry signal?'},
    'lu_change':                    {'id': 'A22', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q: Does persistent urban-to-other landuse loss carry signal?'},
    'rolling_accum_coh':            {'id': 'A23', 'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q4b: Rolling-window matched-filter signal (COH)?'},
    'rolling_accum_card':           {'id': 'A24', 'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q4b: Rolling-window matched-filter signal (CARD)?'},
    'rolling_accum_ms':             {'id': 'A25', 'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q4b: Rolling-window matched-filter signal (MS)?'},
    'block_accum_coh':              {'id': 'A26', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q6b: Block-scope matched-filter signal (COH)?'},
    'block_accum_card':             {'id': 'A27', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q6b: Block-scope matched-filter signal (CARD)?'},
    'block_accum_ms':               {'id': 'A28', 'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q6b: Block-scope matched-filter signal (MS)?'},
    'fusion_ms_card':               {'id': 'F1',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q2: Does MS+CARD beat either alone?',        'composed_of': ['scene_ms', 'scene_card']},
    'fusion_ms_card_cohdrop':       {'id': 'F2',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q2: Full multimodal',                       'composed_of': ['scene_ms', 'scene_card', 'coh_drop']},
    'fusion_card_cohdrop':          {'id': 'F3',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q2: SAR-only multimodal',                   'composed_of': ['scene_card', 'coh_drop']},
    'fusion_ms_cohdrop':            {'id': 'F4',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q2: Optical + COH drop',                    'composed_of': ['scene_ms', 'coh_drop']},
    'fusion_indices_card':          {'id': 'F5',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q2: Indices + CARD',                        'composed_of': ['scene_indices', 'scene_card']},
    'fusion_indices_card_cohdrop':  {'id': 'F6',  'format': 'long', 'join_keys': ['city', 'point_id', 'date'], 'question': 'Q2: Indices + CARD + COH drop',             'composed_of': ['scene_indices', 'scene_card', 'coh_drop']},
    'fusion_composite_cohdrop':     {'id': 'F7',  'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q2: Composite + COH drop',                  'composed_of': ['composite_prepost_bands', 'coh_drop']},
    'fusion_composite_blockstats':  {'id': 'F8',  'format': 'wide', 'join_keys': ['city', 'point_id'],         'question': 'Q6: Dietrich composite + block replication', 'composed_of': ['composite_prepost_bands', 'block_stats']},
}

def scan_disk_and_register(verbose=True):
    """Walk V5_DIR for bda_*_t*.parquet and register any (name, tier) missing
    from MANIFEST_ENTRIES."""
    import pyarrow.parquet as _pq
    pat = re.compile(r'^bda_(.+)_t(\d+)\.parquet$')
    n_added = 0
    n_skip_registered = 0
    n_skip_unknown = 0
    for p in sorted(V5_DIR.glob('bda_*_t*.parquet')):
        m = pat.match(p.name)
        if not m:
            continue
        name = m.group(1)
        tier = int(m.group(2))
        existing = MANIFEST_ENTRIES.get(name)
        if existing and tier in existing.get('tiers_built', []):
            n_skip_registered += 1
            continue
        if name not in PARQUET_CATALOG:
            if verbose:
                print(f"  disk-scan: {p.name} -- no catalog entry, skipping")
            n_skip_unknown += 1
            continue
        meta = PARQUET_CATALOG[name]
        try:
            pf = _pq.ParquetFile(p)
            columns = pf.schema_arrow.names
            n_rows = pf.metadata.num_rows
        except Exception as e:
            if verbose:
                print(f"  disk-scan: {p.name} -- read error: {e}")
            continue
        feat_cols = [c for c in columns if not _is_metadata_column(c)]
        register_manifest(
            name, meta['id'], meta['format'], meta['join_keys'],
            feat_cols, meta['question'], tier, n_rows,
            composed_of=meta.get('composed_of'),
            depends_on_tifs=meta.get('depends_on_tifs'),
        )
        n_added += 1
        if verbose:
            print(f"  disk-scan: registered {p.name} ({n_rows} rows, {len(feat_cols)} features)")
    if verbose:
        print(f"  disk-scan summary: +{n_added} added, {n_skip_registered} already-registered, {n_skip_unknown} unknown-name")
    return n_added

print(f"  PARQUET_CATALOG: {len(PARQUET_CATALOG)} entries, scan_disk_and_register() ready")
print(f"  PROFILE_DIR: {PROFILE_DIR}")


  DATASET_ROOT_V5: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/v5
  Catalog: /mnt/f/PROJECTS/masterthesis/data_stack/bda.sqlite
  nb05bv5_helpers loaded — kernel-based sampling (V5: 5x5, no SAR filter)
    KERNEL_SIZE = 5x5 (25 pixels per point)
    numeric stats: ('mean', 'p10', 'p50', 'p90', 'std', 'min', 'max', 'max_abs_delta')
    categorical detection: filename match against ('landuse', 'lu_at_', 'modal_post_class', 'final_class', 'lu_transition', 'lu_change')
  UNOSAT_ROOT: /content/drive_f/masterthesis/data/unosat_damage_assessments/cities

Cities: 21 across 3 tiers
  Tier 0: 4 cities
    Lysychansk             battle=2022-06-25
    Mariupol               battle=2022-02-24
    Rubizhne               battle=2022-03-04
    Sievierodonetsk        battle=2022-05-05
  Tier 1: 11 cities
    Borodyanka             battle=2022-02-28
    Bucha                  battle=2022-02-27
    Chernihiv              battle=2022-02-24
    Dmytrivka              battle=2022-02-27
    Hostomel     

# CELL 4: A0 -- bda_points (per-tier point identity table)

One row per point (UNOSAT positive, UNOSAT undamaged, sampled building centroid).
All A1..A28 feature parquets join onto this via `point_id`.

In [4]:
# @title CELL 4: bda_points_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('points', tier)
    if not FR_POINTS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_points_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} POINTS ({len(tier_cities)} cities) ---")

    all_pts = []
    for city_name in tier_cities:
        pts, ref_shape, transform = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            print(f"  {city_name}: no points, skip")
            continue

        meta = city_meta[city_name]
        info = meta['manifest']

        df_pts = pts.copy()
        df_pts['tier'] = meta['tier']
        df_pts['battle_start'] = meta['battle_start']
        df_pts['battle_stop'] = meta['battle_stop']
        df_pts['conflict_ongoing'] = meta['battle_stop'] in ('', 'ongoing', None, 'None', 'NaT')
        df_pts['has_card'] = bool(info.get('has_card', False))
        df_pts['has_coh'] = bool(info.get('has_coh', False))
        df_pts['has_ms'] = bool(info.get('has_ms', False))

        all_pts.append(df_pts)
        n_pos = int((df_pts['damage_binary'] == 1).sum())
        n_neg = int((df_pts['damage_binary'] == 0).sum())
        print(f"  {city_name}: {len(df_pts):,} points (pos={n_pos}, neg={n_neg})")

    if not all_pts:
        print(f"  Tier {tier}: no points built")
        continue

    df = pd.concat(all_pts, ignore_index=True)
    save_v5_parquet(df, 'points', tier)
    register_manifest('points', 'A0', 'wide', ['city', 'point_id'], [],
                       'V3 point identity table', tier, len(df))
    print(f"  Saved: bda_points_t{tier} ({len(df):,} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    gc.collect()



--- Tier 0 POINTS (4 cities) ---
  Lysychansk: 7,962 points (pos=1117, neg=6845)
  Mariupol: 22,615 points (pos=3311, neg=19304)
  Rubizhne: 5,002 points (pos=463, neg=4539)
  Sievierodonetsk: 4,418 points (pos=568, neg=3850)
    profile -> bda_points_t0.json + bda_points_t0_features.json (0 features)
  Saved: bda_points_t0 (39,997 rows, 23 cols, 2s)

--- Tier 1 POINTS (11 cities) ---
  Borodyanka: 507 points (pos=68, neg=439)
  Bucha: 1,662 points (pos=194, neg=1468)
  Chernihiv: 2,810 points (pos=358, neg=2452)
  Dmytrivka: 3,136 points (pos=361, neg=2775)
  Hostomel: 4,497 points (pos=618, neg=3879)
  Irpin: 2,749 points (pos=308, neg=2441)
  Makariv: 448 points (pos=61, neg=387)
  Moschun: 962 points (pos=85, neg=877)
  Okhtyrka: 258 points (pos=38, neg=220)
  Trostianets: 143 points (pos=17, neg=126)
  Volnovakha: 1,848 points (pos=174, neg=1674)
    profile -> bda_points_t1.json + bda_points_t1_features.json (0 features)
  Saved: bda_points_t1 (19,020 rows, 23 cols, 2s)

--- Tie

# CELL 5: A1 -- bda_scene_ms (long: per-scene optical)

In [5]:
# @title CELL 5: bda_scene_ms_t{tier}.parquet
MS_BANDS = ['b02', 'b03', 'b04', 'b05', 'b07', 'b08', 'b11', 'b12', 'b8a']
MS_AUX = ['scl', 'cloud_mask', 'visibility']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('scene_ms', tier)
    if not FR_SCENE_MS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_ms_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} SCENE MS ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        ms_dir = STACK_ROOT / city_name / "multispectral" / "flat"
        if not ms_dir.exists():
            continue

        dates = set()
        for f in ms_dir.glob("s2__b02__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m:
                dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])

        for date_str in sorted(dates):
            feats = {}
            for band in MS_BANDS:
                path = ms_dir / f"s2__{band}__{date_str}.tif"
                if path.exists():
                    for _v4_stat, _v4_arr in sample_one_kernel(path, pts, ref_shape).items():
                        feats[_v4_col(f"s2__{band}", _v4_stat)] = _v4_arr
            for aux in MS_AUX:
                path = ms_dir / f"s2__{aux}__{date_str}.tif"
                if path.exists():
                    for _v4_stat, _v4_arr in sample_one_kernel(path, pts, ref_shape).items():
                        feats[_v4_col(f"s2__{aux}", _v4_stat)] = _v4_arr
            fire_dir = STACK_ROOT / city_name / "landuse" / "flat"
            if fire_dir.exists():
                for fire_product in ['fire__active_fire', 'fire__burn_scar']:
                    fp = fire_dir / f"{fire_product}__{date_str}.tif"
                    if fp.exists():
                        for _v4_stat, _v4_arr in sample_one_kernel(fp, pts, ref_shape).items():
                            feats[_v4_col(fire_product, _v4_stat)] = _v4_arr
            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} MS dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'scene_ms', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('scene_ms', 'A1', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q1: Does optical carry per-scene damage signal?', tier, len(df),
                          depends_on_tifs=['s2__b{XX}__{YYYYMMDD}.tif'])
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No MS scenes for tier {tier}")

    gc.collect()



--- Tier 0 SCENE MS ---
  Lysychansk: 11 MS dates
  Mariupol: 12 MS dates
  Rubizhne: 10 MS dates
  Sievierodonetsk: 12 MS dates
    profile -> bda_scene_ms_t0.json + bda_scene_ms_t0_features.json (96 features)
  Saved: bda_scene_ms_t0.parquet (461998 rows, 101 cols, 598s)

--- Tier 1 SCENE MS ---
  Bucha: 10 MS dates
  Chernihiv: 9 MS dates
  Dmytrivka: 9 MS dates
  Hostomel: 7 MS dates
  Irpin: 7 MS dates
  Makariv: 8 MS dates
  Moschun: 7 MS dates
  Okhtyrka: 14 MS dates
  Trostianets: 11 MS dates
  Volnovakha: 12 MS dates
    profile -> bda_scene_ms_t1.json + bda_scene_ms_t1_features.json (96 features)
  Saved: bda_scene_ms_t1.parquet (158535 rows, 101 cols, 240s)

--- Tier 2 SCENE MS ---
  Avdiivka: 25 MS dates
  Chornobaivka: 15 MS dates
  Kharkiv: 14 MS dates
  Kherson: 15 MS dates
  Kramatorsk: 81 MS dates
    profile -> bda_scene_ms_t2.json + bda_scene_ms_t2_features.json (96 features)
  Saved: bda_scene_ms_t2.parquet (74852 rows, 101 cols, 205s)


# CELL 6: A2 -- bda_scene_card (long: per-scene CARD VV/VH)

In [6]:
# @title CELL 6: bda_scene_card_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('scene_card', tier)
    if not FR_SCENE_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} SCENE CARD ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        card_dir = STACK_ROOT / city_name / "SAR_CARD" / "flat"
        if not card_dir.exists():
            continue

        dates = set()
        for f in card_dir.glob("s1__vv__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m:
                dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])

        for date_str in sorted(dates):
            feats = {}
            for pol in ['vv', 'vh']:
                path = card_dir / f"s1__{pol}__{date_str}.tif"
                if not path.exists():
                    cands = list(card_dir.glob(f"s1__{pol}__o*__{date_str}.tif"))
                    if cands:
                        path = cands[0]
                    else:
                        continue
                for _v4_stat, _v4_arr in sample_one_kernel(path, pts, ref_shape).items():
                    feats[_v4_col(f"s1__{pol}", _v4_stat)] = _v4_arr
            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} CARD dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'scene_card', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('scene_card', 'A2', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q1: Does CARD carry per-scene damage signal?', tier, len(df),
                          depends_on_tifs=['s1__vv__{YYYYMMDD}.tif', 's1__vh__{YYYYMMDD}.tif'])
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No CARD scenes for tier {tier}")

    gc.collect()



--- Tier 0 SCENE CARD ---
  Lysychansk: 8 CARD dates
  Mariupol: 14 CARD dates
  Rubizhne: 13 CARD dates
  Sievierodonetsk: 12 CARD dates
    profile -> bda_scene_card_t0.json + bda_scene_card_t0_features.json (16 features)
  Saved: bda_scene_card_t0.parquet (498348 rows, 21 cols, 132s)

--- Tier 1 SCENE CARD ---
  Borodyanka: 10 CARD dates
  Bucha: 9 CARD dates
  Chernihiv: 10 CARD dates
  Dmytrivka: 9 CARD dates
  Hostomel: 10 CARD dates
  Irpin: 10 CARD dates
  Makariv: 9 CARD dates
  Moschun: 10 CARD dates
  Okhtyrka: 10 CARD dates
  Trostianets: 9 CARD dates
  Volnovakha: 8 CARD dates
    profile -> bda_scene_card_t1.json + bda_scene_card_t1_features.json (16 features)
  Saved: bda_scene_card_t1.parquet (181115 rows, 21 cols, 60s)

--- Tier 2 SCENE CARD ---
  Avdiivka: 65 CARD dates
  Chornobaivka: 27 CARD dates
  Kharkiv: 24 CARD dates
  Kherson: 28 CARD dates
  Kramatorsk: 125 CARD dates
  Mykolaiv: 3 CARD dates
    profile -> bda_scene_card_t2.json + bda_scene_card_t2_features

# CELL 7: A3 -- bda_scene_coh (long: per-pair coherence)

In [7]:
# @title CELL 7: bda_scene_coh_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('scene_coh', tier)
    if not FR_SCENE_COH and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_coh_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} SCENE COH ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        coh_dir = STACK_ROOT / city_name / "SAR_SLC" / "flat"
        zscore_dir = STACK_ROOT / city_name / "temporal" / "COH" / "zscore"
        if not coh_dir.exists():
            continue

        coh_pairs = {}
        for f in sorted(coh_dir.glob("*.tif")):
            m = re.match(r's1__coh_(vv|vh)__(?:o\d{3}__)?(\d{8})_(\d{8})\.tif$', f.name)
            if m:
                pol, d1, d2 = m.group(1), m.group(2), m.group(3)
                coh_pairs.setdefault((d1, d2), {})[pol] = f
                continue
            m = re.match(r'COH_(VV|VH)_(PRE|CROSS|POST)_(?:o\d{3}_)?(\d{8})_(\d{8})\.tif$', f.name)
            if m:
                pol, _tag, d1, d2 = m.group(1).lower(), m.group(2), m.group(3), m.group(4)
                coh_pairs.setdefault((d1, d2), {})[pol] = f
                continue

        zscore_map = {}
        if zscore_dir.exists():
            for f in sorted(zscore_dir.glob("*.tif")):
                m = re.search(r'__(\d{8})\.tif$', f.name)
                if m:
                    zscore_map[m.group(1)] = f

        pair_d2 = sorted(set(d2 for (d1, d2) in coh_pairs.keys()))
        ts_map = assign_timesteps(pair_d2, meta['battle_start'])

        for (d1, d2), pol_dict in sorted(coh_pairs.items()):
            feats = {}
            for pol in ['vv', 'vh']:
                path = pol_dict.get(pol)
                if path is None or not path.exists():
                    continue
                for _v4_stat, _v4_arr in sample_one_kernel(path, pts, ref_shape).items():
                    feats[_v4_col(f"s1__coh_{pol}", _v4_stat)] = _v4_arr
            zs_path = zscore_map.get(d2)
            if zs_path and zs_path.exists():
                for _v4_stat, _v4_arr in sample_one_kernel(zs_path, pts, ref_shape).items():
                    feats[_v4_col("s1__coh_vv__zscore", _v4_stat)] = _v4_arr
            if not feats:
                continue

            period = get_period_label(d2, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(d2, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date1'] = d1
            date_df['date2'] = d2
            date_df['date'] = d2
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(coh_pairs)} COH pairs")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'scene_coh', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('scene_coh', 'A3', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q1: Does COH carry per-scene damage signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No COH scenes for tier {tier}")

    gc.collect()



--- Tier 0 SCENE COH ---
  Lysychansk: 5 COH pairs
  Mariupol: 7 COH pairs
  Rubizhne: 5 COH pairs
  Sievierodonetsk: 6 COH pairs
    profile -> bda_scene_coh_t0.json + bda_scene_coh_t0_features.json (24 features)
  Saved: bda_scene_coh_t0.parquet (249633 rows, 31 cols, 74s)

--- Tier 1 SCENE COH ---
  Borodyanka: 9 COH pairs
  Bucha: 8 COH pairs
  Chernihiv: 1 COH pairs
  Dmytrivka: 7 COH pairs
  Hostomel: 9 COH pairs
  Irpin: 9 COH pairs
  Moschun: 9 COH pairs
  Okhtyrka: 9 COH pairs
  Volnovakha: 7 COH pairs
    profile -> bda_scene_coh_t1.json + bda_scene_coh_t1_features.json (24 features)
  Saved: bda_scene_coh_t1.parquet (131751 rows, 31 cols, 53s)

--- Tier 2 SCENE COH ---
  Avdiivka: 63 COH pairs
  Chornobaivka: 10 COH pairs
  Kherson: 10 COH pairs
  Kramatorsk: 52 COH pairs
  Mykolaiv: 1 COH pairs
    profile -> bda_scene_coh_t2.json + bda_scene_coh_t2_features.json (24 features)
  Saved: bda_scene_coh_t2.parquet (87611 rows, 31 cols, 51s)


# CELL 8: A4 -- bda_scene_landuse (long: per-scene landuse class)

In [8]:
# @title CELL 8: bda_scene_landuse_t{tier}.parquet
DATE_RE = re.compile(r'__(\d{8})\.tif$')

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('scene_landuse', tier)
    if not FR_SCENE_LANDUSE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_landuse_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} SCENE LANDUSE ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        flat_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not flat_dir.exists():
            continue

        date_files = {}
        for tif in flat_dir.glob("s2__landuse__*.tif"):
            m = DATE_RE.search(tif.name)
            if m:
                date_files[m.group(1)] = tif

        ts_map = assign_timesteps(sorted(date_files.keys()), meta['battle_start'])

        for date_str in sorted(date_files.keys()):
            tif = date_files[date_str]
            feats = {}
            for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                feats[_v4_col("s2__landuse", _v4_stat)] = _v4_arr

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(date_files)} landuse dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'scene_landuse', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('scene_landuse', 'A4', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q5: Does landuse context help?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No landuse scenes for tier {tier}")

    gc.collect()



--- Tier 0 SCENE LANDUSE ---
  Lysychansk: 14 landuse dates
  Mariupol: 12 landuse dates
  Rubizhne: 13 landuse dates
  Sievierodonetsk: 15 landuse dates
    profile -> bda_scene_landuse_t0.json + bda_scene_landuse_t0_features.json (1 features)
  Saved: bda_scene_landuse_t0.parquet (514144 rows, 6 cols, 10s)

--- Tier 1 SCENE LANDUSE ---
  Borodyanka: 0 landuse dates
  Bucha: 13 landuse dates
  Chernihiv: 11 landuse dates
  Dmytrivka: 11 landuse dates
  Hostomel: 10 landuse dates
  Irpin: 9 landuse dates
  Makariv: 11 landuse dates
  Moschun: 10 landuse dates
  Okhtyrka: 14 landuse dates
  Trostianets: 11 landuse dates
  Volnovakha: 12 landuse dates
    profile -> bda_scene_landuse_t1.json + bda_scene_landuse_t1_features.json (1 features)
  Saved: bda_scene_landuse_t1.parquet (198632 rows, 6 cols, 6s)

--- Tier 2 SCENE LANDUSE ---
  Avdiivka: 28 landuse dates
  Chornobaivka: 19 landuse dates
  Kharkiv: 15 landuse dates
  Kherson: 19 landuse dates
  Kramatorsk: 81 landuse dates
  Mykol

# CELL 9: A5 -- bda_scene_indices (long: per-scene spectral indices)

In [9]:
# @title CELL 9: bda_scene_indices_t{tier}.parquet
KNOWN_INDICES = ['ndvi', 'bsi', 'savi', 'mndwi', 'ndbi', 'ndsi', 'ibi', 'baei', 'ui']
DATE_RE = re.compile(r'__(\d{8})\.tif$')

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('scene_indices', tier)
    if not FR_SCENE_INDICES and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_indices_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} SCENE INDICES ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        flat_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not flat_dir.exists():
            continue

        date_files = {}
        for idx_name in KNOWN_INDICES:
            for tif in flat_dir.glob(f"s2__{idx_name}__*.tif"):
                m = DATE_RE.search(tif.name)
                if m:
                    date_files.setdefault(m.group(1), []).append(tif)
            for tif in flat_dir.glob(f"{idx_name}__*.tif"):
                m = DATE_RE.search(tif.name)
                if m:
                    date_files.setdefault(m.group(1), []).append(tif)

        ts_map = assign_timesteps(sorted(date_files.keys()), meta['battle_start'])

        for date_str in sorted(date_files.keys()):
            feats = {}
            for tif in date_files[date_str]:
                stem = tif.stem
                base = stem[:-(len(date_str)+2)]
                if base.startswith('s2__'):
                    prefix = base
                elif base in KNOWN_INDICES:
                    prefix = f"s2__{base}"
                else:
                    prefix = base
                for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                    feats[_v4_col(prefix, _v4_stat)] = _v4_arr
            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(date_files)} indices dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'scene_indices', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('scene_indices', 'A5', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q1: Do spectral indices outperform raw bands?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No index scenes for tier {tier}")

    gc.collect()



--- Tier 0 SCENE INDICES ---
  Lysychansk: 14 indices dates
  Mariupol: 12 indices dates
  Rubizhne: 13 indices dates
  Sievierodonetsk: 15 indices dates
    profile -> bda_scene_indices_t0.json + bda_scene_indices_t0_features.json (72 features)
  Saved: bda_scene_indices_t0.parquet (514144 rows, 77 cols, 611s)

--- Tier 1 SCENE INDICES ---
  Borodyanka: 0 indices dates
  Bucha: 13 indices dates
  Chernihiv: 11 indices dates
  Dmytrivka: 11 indices dates
  Hostomel: 10 indices dates
  Irpin: 9 indices dates
  Makariv: 11 indices dates
  Moschun: 10 indices dates
  Okhtyrka: 14 indices dates
  Trostianets: 11 indices dates
  Volnovakha: 12 indices dates
    profile -> bda_scene_indices_t1.json + bda_scene_indices_t1_features.json (72 features)
  Saved: bda_scene_indices_t1.parquet (198632 rows, 77 cols, 271s)

--- Tier 2 SCENE INDICES ---
  Avdiivka: 28 indices dates
  Chornobaivka: 19 indices dates
  Kharkiv: 15 indices dates
  Kherson: 19 indices dates
  Kramatorsk: 81 indices dates


# CELL 10: A6 -- bda_scene_nbr (long: per-scene NBR)

In [10]:
# @title CELL 10: bda_scene_nbr_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('scene_nbr', tier)
    if not FR_SCENE_NBR and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_nbr_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} SCENE NBR ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        ms_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not ms_dir.exists():
            continue

        dates = set()
        for f in ms_dir.glob("s2__nbr__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m:
                dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])

        for date_str in sorted(dates):
            path = ms_dir / f"s2__nbr__{date_str}.tif"
            if not path.exists():
                continue
            feats = {}
            for _v4_stat, _v4_arr in sample_one_kernel(path, pts, ref_shape).items():
                feats[_v4_col("s2__nbr", _v4_stat)] = _v4_arr

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} NBR dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'scene_nbr', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('scene_nbr', 'A6', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q1: Does NBR carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No NBR scenes for tier {tier}")

    gc.collect()



--- Tier 0 SCENE NBR ---
  Lysychansk: 14 NBR dates
  Mariupol: 12 NBR dates
  Rubizhne: 13 NBR dates
  Sievierodonetsk: 15 NBR dates
    profile -> bda_scene_nbr_t0.json + bda_scene_nbr_t0_features.json (8 features)
  Saved: bda_scene_nbr_t0.parquet (514144 rows, 13 cols, 68s)

--- Tier 1 SCENE NBR ---
  Borodyanka: 0 NBR dates
  Bucha: 13 NBR dates
  Chernihiv: 11 NBR dates
  Dmytrivka: 11 NBR dates
  Hostomel: 10 NBR dates
  Irpin: 9 NBR dates
  Makariv: 11 NBR dates
  Moschun: 10 NBR dates
  Okhtyrka: 14 NBR dates
  Trostianets: 11 NBR dates
  Volnovakha: 12 NBR dates
    profile -> bda_scene_nbr_t1.json + bda_scene_nbr_t1_features.json (8 features)
  Saved: bda_scene_nbr_t1.parquet (198632 rows, 13 cols, 31s)

--- Tier 2 SCENE NBR ---
  Avdiivka: 28 NBR dates
  Chornobaivka: 19 NBR dates
  Kharkiv: 15 NBR dates
  Kherson: 19 NBR dates
  Kramatorsk: 81 NBR dates
  Mykolaiv: 0 NBR dates
    profile -> bda_scene_nbr_t2.json + bda_scene_nbr_t2_features.json (8 features)
  Saved: bda_

# CELL 11: A7 -- bda_rolling_coh (long: rolling-window coherence, all windows)

In [11]:
# @title CELL 11: bda_rolling_coh_t{tier}.parquet
ROLL_WINDOWS = ROLLING_WINDOW_SIZES  # [3, 7, 13]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_coh', tier)
    if not FR_ROLLING_COH and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_rolling_coh_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING COH ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        roll_dir = STACK_ROOT / city_name / "temporal" / "COH" / "rolling"
        if not roll_dir.exists():
            continue

        all_dates = set()
        for ws in ROLL_WINDOWS:
            for f in roll_dir.glob(f"s1__coh_vv__roll{ws}__*.tif"):
                m = re.search(r'__(\d{8})\.tif$', f.name)
                if m:
                    all_dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(all_dates), meta['battle_start'])

        for date_str in sorted(all_dates):
            feats = {}
            for ws in ROLL_WINDOWS:
                path = roll_dir / f"s1__coh_vv__roll{ws}__{date_str}.tif"
                if path.exists():
                    for _v4_stat, _v4_arr in sample_one_kernel(path, pts, ref_shape).items():
                        feats[_v4_col(f"s1__coh_vv__roll{ws}", _v4_stat)] = _v4_arr
            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(all_dates)} rolling COH dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'rolling_coh', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_coh', 'A7', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q4: Do rolling windows add value (COH)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling COH for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING COH ---
  Lysychansk: 3 rolling COH dates
  Mariupol: 5 rolling COH dates
  Rubizhne: 3 rolling COH dates
  Sievierodonetsk: 4 rolling COH dates
    profile -> bda_rolling_coh_t0.json + bda_rolling_coh_t0_features.json (16 features)
  Saved: bda_rolling_coh_t0.parquet (169639 rows, 21 cols, 23s)

--- Tier 1 ROLLING COH ---
  Borodyanka: 7 rolling COH dates
  Bucha: 6 rolling COH dates
  Dmytrivka: 5 rolling COH dates
  Hostomel: 7 rolling COH dates
  Irpin: 7 rolling COH dates
  Moschun: 7 rolling COH dates
  Okhtyrka: 7 rolling COH dates
  Volnovakha: 5 rolling COH dates
    profile -> bda_rolling_coh_t1.json + bda_rolling_coh_t1_features.json (16 features)
  Saved: bda_rolling_coh_t1.parquet (97703 rows, 21 cols, 19s)

--- Tier 2 ROLLING COH ---
  Avdiivka: 61 rolling COH dates
  Chornobaivka: 8 rolling COH dates
  Kherson: 8 rolling COH dates
  Kramatorsk: 50 rolling COH dates
    profile -> bda_rolling_coh_t2.json + bda_rolling_coh_t2_features.json (24 features)

# CELL 12: A8 -- bda_rolling_card (long: rolling-window CARD, all windows)

In [12]:
# @title CELL 12: bda_rolling_card_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_card', tier)
    if not FR_ROLLING_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_rolling_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING CARD ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]
        roll_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "rolling"
        if not roll_dir.exists():
            continue

        all_dates = set()
        for ws in ROLL_WINDOWS:
            for f in roll_dir.glob(f"s1__vv__roll{ws}__*.tif"):
                m = re.search(r'__(\d{8})\.tif$', f.name)
                if m:
                    all_dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(all_dates), meta['battle_start'])

        for date_str in sorted(all_dates):
            feats = {}
            for ws in ROLL_WINDOWS:
                for pol in ['vv', 'vh']:
                    path = roll_dir / f"s1__{pol}__roll{ws}__{date_str}.tif"
                    if path.exists():
                        for _v4_stat, _v4_arr in sample_one_kernel(path, pts, ref_shape).items():
                            feats[_v4_col(f"s1__{pol}__roll{ws}", _v4_stat)] = _v4_arr
            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(all_dates)} rolling CARD dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'rolling_card', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_card', 'A8', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q4: Do rolling windows add value (CARD)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling CARD for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING CARD ---
  Lysychansk: 6 rolling CARD dates
  Mariupol: 12 rolling CARD dates
  Rubizhne: 11 rolling CARD dates
  Sievierodonetsk: 10 rolling CARD dates
    profile -> bda_rolling_card_t0.json + bda_rolling_card_t0_features.json (48 features)
  Saved: bda_rolling_card_t0.parquet (418354 rows, 53 cols, 192s)

--- Tier 1 ROLLING CARD ---
  Borodyanka: 8 rolling CARD dates
  Bucha: 7 rolling CARD dates
  Chernihiv: 8 rolling CARD dates
  Dmytrivka: 7 rolling CARD dates
  Hostomel: 8 rolling CARD dates
  Irpin: 8 rolling CARD dates
  Makariv: 7 rolling CARD dates
  Moschun: 8 rolling CARD dates
  Okhtyrka: 8 rolling CARD dates
  Trostianets: 7 rolling CARD dates
  Volnovakha: 6 rolling CARD dates
    profile -> bda_rolling_card_t1.json + bda_rolling_card_t1_features.json (32 features)
  Saved: bda_rolling_card_t1.parquet (143075 rows, 37 cols, 70s)

--- Tier 2 ROLLING CARD ---
  Avdiivka: 63 rolling CARD dates
  Chornobaivka: 26 rolling CARD dates
  Kharkiv: 22 rolling 

# CELL 13: A9 -- bda_composite_prepost_bands (wide: pre vs post composite + deltas)

In [13]:
# @title CELL 13: bda_composite_prepost_bands_t{tier}.parquet
# One row per point. One column per (composite_TIF, period). Plus delta columns.

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('composite_prepost_bands', tier)
    if not FR_COMPOSITE_PREPOST_BANDS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_prepost_bands_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE PREPOST BANDS ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        feats = {}
        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        if not comp_dir.exists():
            continue

        for period_dir in sorted(comp_dir.iterdir()):
            if not period_dir.is_dir():
                if period_dir.suffix == '.tif' and 'landuse' not in period_dir.stem.lower():
                    for _v4_stat, _v4_arr in sample_one_kernel(period_dir, pts, ref_shape).items():
                        feats[_v4_col(period_dir.stem, _v4_stat)] = _v4_arr
                continue

            period = period_dir.name
            for tif in sorted(period_dir.glob("*.tif")):
                stem = tif.stem
                if 'landuse' in stem.lower():
                    continue  # landuse goes to A10

                if stem.startswith('qa__'):
                    col = f"{stem}__{period}"
                elif stem.startswith('s2__'):
                    col = f"{stem}__{period}"
                elif stem.startswith('composite_'):
                    band = stem.replace('composite_', '')
                    col = f"s2__{band}__{period}"
                elif stem.startswith('visibility_'):
                    col = f"vis__{stem}__{period}"
                else:
                    col = f"{period}__{stem}"

                for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():

                    feats[_v4_col(col, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)

        # delta columns: post_winter - prebattle / winter
        delta_count = 0
        pre_patterns = [
            ('__prebattle_baseline', '__post_winter_baseline'),
            ('__winter_baseline', '__post_winter_baseline'),
        ]
        for pre_tag, post_tag in pre_patterns:
            pre_cols = [c for c in df.columns if c.endswith(pre_tag) and df[c].dtype.kind in ('f', 'i', 'u')]
            for pre_col in pre_cols:
                post_col = pre_col[:-len(pre_tag)] + post_tag
                if post_col in df.columns:
                    delta_name = pre_col[:-len(pre_tag)] + '__delta'
                    if delta_name not in df.columns:
                        df[delta_name] = df[post_col] - df[pre_col]
                        delta_count += 1

        df = impute_wide_parquet(df, 'composite')
        out = save_v5_parquet(df, 'composite_prepost_bands', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('composite_prepost_bands', 'A9', 'wide', ['city', 'point_id'], feat_cols,
                          'Q3: Do composites work? (Dietrich baseline)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {delta_count} deltas, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite data for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE PREPOST BANDS ---
  Lysychansk: 504 feature columns
  Mariupol: 504 feature columns
  Rubizhne: 504 feature columns
  Sievierodonetsk: 504 feature columns
    impute_wide_parquet(composite): 512 NaN -> 0 NaN, was_observed_composite added
    profile -> bda_composite_prepost_bands_t0.json + bda_composite_prepost_bands_t0_features.json (504 features)
  Saved: bda_composite_prepost_bands_t0.parquet (39997 rows, 507 cols, 0 deltas, 323s)

--- Tier 1 COMPOSITE PREPOST BANDS ---
  Bucha: 504 feature columns
  Chernihiv: 504 feature columns
  Dmytrivka: 376 feature columns
  Hostomel: 504 feature columns
  Irpin: 376 feature columns
  Makariv: 376 feature columns
  Moschun: 504 feature columns
  Okhtyrka: 504 feature columns
  Trostianets: 504 feature columns
  Volnovakha: 464 feature columns
    impute_wide_parquet(composite): 968528 NaN -> 0 NaN, was_observed_composite added
    profile -> bda_composite_prepost_bands_t1.json + bda_composite_prepost_bands_t1_features.js

# CELL 14: A10 -- bda_composite_prepost_landuse (wide: pre vs post composite landuse)

In [14]:
# @title CELL 14: bda_composite_prepost_landuse_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('composite_prepost_landuse', tier)
    if not FR_COMPOSITE_PREPOST_LANDUSE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_prepost_landuse_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE PREPOST LANDUSE ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        feats = {}
        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        if not comp_dir.exists():
            continue

        for period_dir in sorted(comp_dir.iterdir()):
            if not period_dir.is_dir():
                continue
            period = period_dir.name
            for tif in sorted(period_dir.glob("*landuse*.tif")):
                col = f"landuse__{period}"
                for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                    feats[_v4_col(col, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name

        # landuse_changed flag: pre vs post class differ?
        pre_cols = [c for c in city_df.columns if 'prebattle' in c or 'pre_winter' in c or 'baseline' in c]
        post_cols = [c for c in city_df.columns if 'postbattle' in c or 'post_winter' in c or 'assessment' in c]
        if pre_cols and post_cols:
            pre_v = city_df[pre_cols[0]] if len(pre_cols) == 1 else city_df[pre_cols].mode(axis=1).iloc[:, 0]
            post_v = city_df[post_cols[0]] if len(post_cols) == 1 else city_df[post_cols].mode(axis=1).iloc[:, 0]
            city_df['landuse_changed'] = (pre_v != post_v).astype(int)

        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'composite_landuse')
        out = save_v5_parquet(df, 'composite_prepost_landuse', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('composite_prepost_landuse', 'A10', 'wide', ['city', 'point_id'], feat_cols,
                          'Q5: Does composite landuse change help?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite landuse for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE PREPOST LANDUSE ---
  Lysychansk: 3 feature columns
  Mariupol: 3 feature columns
  Rubizhne: 3 feature columns
  Sievierodonetsk: 3 feature columns
    impute_wide_parquet(composite_landuse): 4 NaN -> 0 NaN, was_observed_composite_landuse added
    profile -> bda_composite_prepost_landuse_t0.json + bda_composite_prepost_landuse_t0_features.json (4 features)
  Saved: bda_composite_prepost_landuse_t0.parquet (39997 rows, 7 cols, 8s)

--- Tier 1 COMPOSITE PREPOST LANDUSE ---
  Bucha: 3 feature columns
  Chernihiv: 3 feature columns
  Dmytrivka: 2 feature columns
  Hostomel: 3 feature columns
  Irpin: 2 feature columns
  Makariv: 2 feature columns
  Moschun: 3 feature columns
  Okhtyrka: 3 feature columns
  Trostianets: 3 feature columns
  Volnovakha: 3 feature columns
    impute_wide_parquet(composite_landuse): 13301 NaN -> 0 NaN, was_observed_composite_landuse added
    profile -> bda_composite_prepost_landuse_t1.json + bda_composite_prepost_landuse_t1_features.jso

# CELL 15: A11 -- bda_composite_vs_scenes_bands (long: pre-composite vs post-scene delta)

In [15]:
# @title CELL 15: bda_composite_vs_scenes_bands_t{tier}.parquet
# For each post-battle scene: scene value + delta from pre-composite (per band)
MS_BANDS = ['b02', 'b03', 'b04', 'b05', 'b07', 'b08', 'b11', 'b12', 'b8a']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('composite_vs_scenes_bands', tier)
    if not FR_COMPOSITE_VS_SCENES_BANDS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_vs_scenes_bands_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE VS SCENES BANDS ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]

        # Pre-composite band values (one array per band)
        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        pre_band_vals = {}  # band -> 1D array of point values
        if comp_dir.exists():
            for period_dir in sorted(comp_dir.iterdir()):
                if not period_dir.is_dir():
                    continue
                pname = period_dir.name
                if 'pre' not in pname.lower() and 'baseline' not in pname.lower():
                    continue
                for tif in sorted(period_dir.glob("*.tif")):
                    if 'landuse' in tif.stem.lower():
                        continue
                    stem = tif.stem
                    # Identify band: split __ and find a known band name
                    parts = stem.replace('composite_', 's2__').split('__')
                    band = None
                    for p in parts:
                        if p in MS_BANDS:
                            band = p
                            break
                    if band is None:
                        continue
                    if band not in pre_band_vals:  # first matching pre composite per band wins
                        for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                            pre_band_vals[_v4_col(band, _v4_stat)] = _v4_arr
        if not pre_band_vals:
            print(f"  {city_name}: no pre-composite, skip")
            continue

        # Iterate post-battle MS scenes
        ms_dir = STACK_ROOT / city_name / "multispectral" / "flat"
        if not ms_dir.exists():
            continue

        post_dates = set()
        for f in ms_dir.glob("s2__b02__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m and get_period_label(m.group(1), meta['battle_start'], meta['battle_stop']) == 'postbattle':
                post_dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(post_dates), meta['battle_start'])

        for date_str in sorted(post_dates):
            feats = {}
            for band in MS_BANDS:
                path = ms_dir / f"s2__{band}__{date_str}.tif"
                if not path.exists():
                    continue
                # V4: compute per-kernel stats for the scene; compute per-stat
                # delta against the matching pre-composite stat.
                scene_dict = sample_one_kernel(path, pts, ref_shape)
                for _v4_stat, _v4_arr in scene_dict.items():
                    feats[_v4_col(f"scene_s2__{band}", _v4_stat)] = _v4_arr
                    pre_key = _v4_col(band, _v4_stat)
                    if pre_key in pre_band_vals:
                        feats[_v4_col(f"delta_s2__{band}", _v4_stat)] = _v4_arr - pre_band_vals[pre_key]

            if not feats:
                continue

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = ts_map.get(date_str, 0)
            date_df['period_label'] = 'postbattle'
            rows.append(date_df)

        print(f"  {city_name}: {len(post_dates)} post-battle dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'composite_vs_scenes_bands', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('composite_vs_scenes_bands', 'A11', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q3: Does per-scene trajectory beat static snapshot?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite-vs-scenes data for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE VS SCENES BANDS ---
  Lysychansk: no pre-composite, skip
  Mariupol: no pre-composite, skip
  Rubizhne: no pre-composite, skip
  Sievierodonetsk: no pre-composite, skip
  No composite-vs-scenes data for tier 0

--- Tier 1 COMPOSITE VS SCENES BANDS ---
  Borodyanka: no pre-composite, skip
  Bucha: no pre-composite, skip
  Chernihiv: no pre-composite, skip
  Dmytrivka: no pre-composite, skip
  Hostomel: no pre-composite, skip
  Irpin: no pre-composite, skip
  Makariv: no pre-composite, skip
  Moschun: no pre-composite, skip
  Okhtyrka: no pre-composite, skip
  Trostianets: no pre-composite, skip
  Volnovakha: no pre-composite, skip
  No composite-vs-scenes data for tier 1

--- Tier 2 COMPOSITE VS SCENES BANDS ---
  Avdiivka: no pre-composite, skip
  Chornobaivka: no pre-composite, skip
  Kharkiv: no pre-composite, skip
  Kherson: no pre-composite, skip
  Kramatorsk: no pre-composite, skip
  Mykolaiv: no pre-composite, skip
  No composite-vs-scenes data for tier 2


# CELL 16: A12 -- bda_composite_vs_scenes_landuse (long: pre-composite vs post-scene landuse)

In [16]:
# @title CELL 16: bda_composite_vs_scenes_landuse_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('composite_vs_scenes_landuse', tier)
    if not FR_COMPOSITE_VS_SCENES_LANDUSE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_vs_scenes_landuse_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE VS SCENES LANDUSE ---")
    rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        meta = city_meta[city_name]

        # Pre-composite landuse
        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        pre_lu_vals = None
        if comp_dir.exists():
            for period_dir in sorted(comp_dir.iterdir()):
                if not period_dir.is_dir():
                    continue
                pname = period_dir.name
                if 'pre' not in pname.lower() and 'baseline' not in pname.lower():
                    continue
                for tif in sorted(period_dir.glob("*landuse*.tif")):
                    _v4_dict = sample_one_kernel(tif, pts, ref_shape)
                    pre_lu_vals = _v4_dict.get('mode', _v4_dict.get('mean'))
                    break
                if pre_lu_vals is not None:
                    break

        if pre_lu_vals is None:
            print(f"  {city_name}: no pre-composite landuse, skip")
            continue

        flat_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not flat_dir.exists():
            continue

        post_dates = set()
        for tif in flat_dir.glob("s2__landuse__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', tif.name)
            if m and get_period_label(m.group(1), meta['battle_start'], meta['battle_stop']) == 'postbattle':
                post_dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(post_dates), meta['battle_start'])

        for date_str in sorted(post_dates):
            tif_path = flat_dir / f"s2__landuse__{date_str}.tif"
            if not tif_path.exists():
                continue
            _v4_dict = sample_one_kernel(tif_path, pts, ref_shape)
            post_lu_vals = _v4_dict.get('mode', _v4_dict.get('mean'))
            feats = {
                'pre_landuse': pre_lu_vals,
                'post_landuse': post_lu_vals,
                'landuse_changed': ((pre_lu_vals != post_lu_vals)
                                     & (~np.isnan(pre_lu_vals)) & (~np.isnan(post_lu_vals))).astype(int),
            }

            date_df = pd.DataFrame(feats)
            date_df['point_id'] = pts['point_id'].tolist()
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = ts_map.get(date_str, 0)
            date_df['period_label'] = 'postbattle'
            rows.append(date_df)

        print(f"  {city_name}: {len(post_dates)} post-battle landuse dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v5_parquet(df, 'composite_vs_scenes_landuse', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('composite_vs_scenes_landuse', 'A12', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q5: Per-scene landuse trajectory?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite-vs-scenes landuse for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE VS SCENES LANDUSE ---
  Lysychansk: 6 post-battle landuse dates
  Mariupol: 7 post-battle landuse dates
  Rubizhne: 6 post-battle landuse dates
  Sievierodonetsk: 8 post-battle landuse dates
    profile -> bda_composite_vs_scenes_landuse_t0.json + bda_composite_vs_scenes_landuse_t0_features.json (3 features)
  Saved: bda_composite_vs_scenes_landuse_t0.parquet (271433 rows, 8 cols, 6s)

--- Tier 1 COMPOSITE VS SCENES LANDUSE ---
  Borodyanka: no pre-composite landuse, skip
  Bucha: 4 post-battle landuse dates
  Chernihiv: 5 post-battle landuse dates
  Dmytrivka: 2 post-battle landuse dates
  Hostomel: 4 post-battle landuse dates
  Irpin: 2 post-battle landuse dates
  Makariv: 4 post-battle landuse dates
  Moschun: 4 post-battle landuse dates
  Okhtyrka: 6 post-battle landuse dates
  Trostianets: 5 post-battle landuse dates
  Volnovakha: 6 post-battle landuse dates
    profile -> bda_composite_vs_scenes_landuse_t1.json + bda_composite_vs_scenes_landuse_t1_features.j

# CELL 17: A13 -- bda_prepost_single_card (wide: single-scene CARD pre/post)

In [17]:
# @title CELL 17: bda_prepost_single_card_t{tier}.parquet
# Single-scene CARD pre + post + delta per polarization

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('prepost_single_card', tier)
    if not FR_PREPOST_SINGLE_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  prepost_single_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} PREPOST SINGLE CARD ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        feats = {}
        card_dir = STACK_ROOT / city_name / "SAR_CARD" / SAR_CARD_PREPOST_SUBDIR
        card_meta_path = card_dir / "card_prepost_meta.json"
        pre_date_card = None
        post_date_card = None

        if card_dir.exists():
            if card_meta_path.exists():
                with open(card_meta_path) as f:
                    card_meta = json.load(f)
                pre_date_card = card_meta.get('vv_pre_date')
                post_date_card = card_meta.get('vv_post_date')

            for pol in ['vv', 'vh']:
                for phase in ['pre', 'post', 'delta']:
                    if phase != 'delta':
                        pattern = f"s1__{pol}__card_prepost__{phase}__*.tif"
                    else:
                        pattern = f"s1__{pol}__card_prepost__delta.tif"
                    matches = list(card_dir.glob(pattern))
                    if not matches:
                        continue
                    for _v4_stat, _v4_arr in sample_one_kernel(matches[0], pts, ref_shape).items():
                        feats[_v4_col(f"{phase}_{pol}", _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        if pre_date_card:
            city_df['pre_date_card'] = pre_date_card
        if post_date_card:
            city_df['post_date_card'] = post_date_card

        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feats, card={pre_date_card}->{post_date_card}")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'card_prepost')
        out = save_v5_parquet(df, 'prepost_single_card', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('prepost_single_card', 'A13', 'wide', ['city', 'point_id'], feat_cols,
                          'Q3: Single-scene CARD pre/post', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No single CARD prepost for tier {tier}")

    gc.collect()



--- Tier 0 PREPOST SINGLE CARD ---
  Lysychansk: 48 feats, card=20220608->20220726
  Mariupol: 48 feats, card=20220204->20220604
  Rubizhne: 48 feats, card=20220211->20220530
  Sievierodonetsk: 48 feats, card=20220412->20220717
    impute_wide_parquet(card_prepost): 0 NaN -> 0 NaN, was_observed_card_prepost added
    profile -> bda_prepost_single_card_t0.json + bda_prepost_single_card_t0_features.json (48 features)
  Saved: bda_prepost_single_card_t0.parquet (39997 rows, 53 cols, 32s)

--- Tier 1 PREPOST SINGLE CARD ---
  Borodyanka: 48 feats, card=20220207->20220420
  Bucha: 48 feats, card=20220212->20220507
  Chernihiv: 48 feats, card=20220130->20220424
  Dmytrivka: 48 feats, card=20220212->20220507
  Hostomel: 48 feats, card=20220207->20220420
  Irpin: 48 feats, card=20220207->20220420
  Makariv: 48 feats, card=20220209->20220410
  Moschun: 48 feats, card=20220207->20220420
  Okhtyrka: 48 feats, card=20220209->20220422
  Trostianets: 48 feats, card=20220209->20220410
  Volnovakha: 

# CELL 18: A14 -- bda_coh_drop (wide: COH drop accumulator)

In [18]:
# @title CELL 18: bda_coh_drop_t{tier}.parquet
COH_DROP_PRODUCTS = [
    's1__coh__running_min.tif',
    's1__coh__drop_count.tif',
    's1__coh__date_first_drop.tif',
    's1__coh__date_worst_drop.tif',
    's1__coh__max_drop.tif',
    's1__coh__scenes_observed.tif',
    's1__coh__lu_transition.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('coh_drop', tier)
    if not FR_COH_DROP and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  coh_drop_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COH DROP ACCUMULATOR ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        drop_dir = STACK_ROOT / city_name / "temporal" / "COH" / "coh_drop_accumulator"
        if not drop_dir.exists():
            print(f"  {city_name}: no coh_drop_accumulator, skip")
            continue

        feats = {}
        for tif_name in COH_DROP_PRODUCTS:
            tif_path = drop_dir / tif_name
            if tif_path.exists():
                for _v4_stat, _v4_arr in sample_one_kernel(tif_path, pts, ref_shape).items():
                    feats[_v4_col(tif_path.stem, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'cohdrop')
        out = save_v5_parquet(df, 'coh_drop', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('coh_drop', 'A14', 'wide', ['city', 'point_id'], feat_cols,
                          'Q1: Does cumulative COH drop carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No COH drop data for tier {tier}")

    gc.collect()



--- Tier 0 COH DROP ACCUMULATOR ---
  Lysychansk: 49 feature columns
  Mariupol: 49 feature columns
  Rubizhne: no coh_drop_accumulator, skip
  Sievierodonetsk: 49 feature columns
    impute_wide_parquet(cohdrop): 1169404 NaN -> 0 NaN, was_observed_cohdrop added
    profile -> bda_coh_drop_t0.json + bda_coh_drop_t0_features.json (49 features)
  Saved: bda_coh_drop_t0.parquet (34995 rows, 52 cols, 8s)

--- Tier 1 COH DROP ACCUMULATOR ---
  Borodyanka: 49 feature columns
  Bucha: 49 feature columns
  Chernihiv: no coh_drop_accumulator, skip
  Dmytrivka: 49 feature columns
  Hostomel: 49 feature columns
  Irpin: 49 feature columns
  Makariv: no coh_drop_accumulator, skip
  Moschun: 49 feature columns
  Okhtyrka: 49 feature columns
  Trostianets: no coh_drop_accumulator, skip
  Volnovakha: 49 feature columns
    impute_wide_parquet(cohdrop): 579917 NaN -> 0 NaN, was_observed_cohdrop added
    profile -> bda_coh_drop_t1.json + bda_coh_drop_t1_features.json (49 features)
  Saved: bda_coh_dr

# CELL 19: A15 -- bda_block_stats (wide: Dietrich CARD/COH baselines + block stats)

In [19]:
# @title CELL 19: bda_block_stats_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('block_stats', tier)
    if not FR_BLOCK_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  block_stats_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK STATS ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        feats = {}

        # CARD baseline
        bl_card_dir = STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats"
        if bl_card_dir.exists():
            for tif in sorted(bl_card_dir.glob("s1__*__baseline__*.tif")):
                for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                    feats[_v4_col(tif.stem, _v4_stat)] = _v4_arr
        # COH baseline
        bl_coh_dir = STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"
        if bl_coh_dir.exists():
            for tif in sorted(bl_coh_dir.glob("*.tif")):
                for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                    feats[_v4_col(tif.stem, _v4_stat)] = _v4_arr
        # COH post-battle baseline
        coh_post_dir = STACK_ROOT / city_name / "temporal" / "COH" / "post_baseline"
        if coh_post_dir.exists():
            for tif in sorted(coh_post_dir.glob("*.tif")):
                for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                    feats[_v4_col(tif.stem, _v4_stat)] = _v4_arr
        # Block stats
        bs_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "block_stats"
        if bs_dir.exists():
            for tif in sorted(bs_dir.glob("s1__*.tif")):
                for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                    feats[_v4_col(tif.stem, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'blockstats')
        out = save_v5_parquet(df, 'block_stats', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('block_stats', 'A15', 'wide', ['city', 'point_id'], feat_cols,
                          'Q6: Does Dietrich block approach work with GroupKFold?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block stats for tier {tier}")

    gc.collect()



--- Tier 0 BLOCK STATS ---
  Lysychansk: 640 feature columns
  Mariupol: 848 feature columns
  Rubizhne: 592 feature columns
  Sievierodonetsk: 592 feature columns
    impute_wide_parquet(blockstats): 9643066 NaN -> 6399520 NaN, was_observed_blockstats added
    profile -> bda_block_stats_t0.json + bda_block_stats_t0_features.json (848 features)
  Saved: bda_block_stats_t0.parquet (39997 rows, 851 cols, 398s)

--- Tier 1 BLOCK STATS ---
  Borodyanka: 480 feature columns
  Bucha: 480 feature columns
  Chernihiv: 384 feature columns
  Dmytrivka: 480 feature columns
  Hostomel: 480 feature columns
  Irpin: 480 feature columns
  Makariv: 384 feature columns
  Moschun: 480 feature columns
  Okhtyrka: 480 feature columns
  Trostianets: 384 feature columns
  Volnovakha: 480 feature columns
    impute_wide_parquet(blockstats): 1064304 NaN -> 0 NaN, was_observed_blockstats added
    profile -> bda_block_stats_t1.json + bda_block_stats_t1_features.json (480 features)
  Saved: bda_block_stats_t1

# CELL 20: A16 -- bda_rolling_stats_roll3 (wide: rolling-stats sample at points, window=3)

In [20]:
# @title CELL 20: bda_rolling_stats_roll3_t{tier}.parquet

def _sample_dir_at_points(pts, ref_shape, tif_dir, prefix_filter=None):
    feats = {}
    if not tif_dir.exists():
        return feats
    for tif in sorted(tif_dir.glob("*.tif")):
        stem = tif.stem
        if prefix_filter and not any(stem.startswith(p) for p in prefix_filter):
            continue
        for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
            feats[_v4_col(stem, _v4_stat)] = _v4_arr
    return feats

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_stats_roll3', tier)
    if not FR_ROLLING_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_stats_roll3_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING STATS ROLL3 ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        feats = {}
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats",
            prefix_filter=["s1__vv__baseline", "s1__vh__baseline"]))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_stats",
            prefix_filter=["s1__vv__roll3__", "s1__vh__roll3__"]))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "temporal" / "COH" / "rolling_stats",
            prefix_filter=["s1__coh_vv__roll3__"]))

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'rolling_stats')
        out = save_v5_parquet(df, 'rolling_stats_roll3', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_stats_roll3', 'A16', 'wide', ['city', 'point_id'], feat_cols,
                          'Q4: Rolling stats (window=3)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling stats roll3 for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING STATS ROLL3 ---
  Lysychansk: 304 feature columns
  Mariupol: 304 feature columns
  Rubizhne: 256 feature columns
  Sievierodonetsk: 304 feature columns
    impute_wide_parquet(rolling_stats): 4998088 NaN -> 4479664 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll3_t0.json + bda_rolling_stats_roll3_t0_features.json (304 features)
  Saved: bda_rolling_stats_roll3_t0.parquet (39997 rows, 307 cols, 115s)

--- Tier 1 ROLLING STATS ROLL3 ---
  Borodyanka: 304 feature columns
  Bucha: 304 feature columns
  Chernihiv: 256 feature columns
  Dmytrivka: 304 feature columns
  Hostomel: 304 feature columns
  Irpin: 304 feature columns
  Makariv: 256 feature columns
  Moschun: 304 feature columns
  Okhtyrka: 304 feature columns
  Trostianets: 256 feature columns
  Volnovakha: 304 feature columns
    impute_wide_parquet(rolling_stats): 2546668 NaN -> 2130240 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll3_t1.json + bda_rolli

# CELL 21: A17 -- bda_rolling_stats_roll7 (wide: rolling-stats sample at points, window=7)

In [21]:
# @title CELL 21: bda_rolling_stats_roll7_t{tier}.parquet

def _sample_dir_at_points(pts, ref_shape, tif_dir, prefix_filter=None):
    feats = {}
    if not tif_dir.exists():
        return feats
    for tif in sorted(tif_dir.glob("*.tif")):
        stem = tif.stem
        if prefix_filter and not any(stem.startswith(p) for p in prefix_filter):
            continue
        for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
            feats[_v4_col(stem, _v4_stat)] = _v4_arr
    return feats

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_stats_roll7', tier)
    if not FR_ROLLING_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_stats_roll7_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING STATS ROLL7 ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        feats = {}
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats",
            prefix_filter=["s1__vv__baseline", "s1__vh__baseline"]))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_stats",
            prefix_filter=["s1__vv__roll7__", "s1__vh__roll7__"]))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "temporal" / "COH" / "rolling_stats",
            prefix_filter=["s1__coh_vv__roll7__"]))

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'rolling_stats')
        out = save_v5_parquet(df, 'rolling_stats_roll7', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_stats_roll7', 'A17', 'wide', ['city', 'point_id'], feat_cols,
                          'Q4: Rolling stats (window=7)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling stats roll7 for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING STATS ROLL7 ---
  Lysychansk: 176 feature columns
  Mariupol: 176 feature columns
  Rubizhne: 128 feature columns
  Sievierodonetsk: 176 feature columns
    impute_wide_parquet(rolling_stats): 518424 NaN -> 0 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll7_t0.json + bda_rolling_stats_roll7_t0_features.json (176 features)
  Saved: bda_rolling_stats_roll7_t0.parquet (39997 rows, 179 cols, 104s)

--- Tier 1 ROLLING STATS ROLL7 ---
  Borodyanka: 176 feature columns
  Bucha: 176 feature columns
  Chernihiv: 128 feature columns
  Dmytrivka: 176 feature columns
  Hostomel: 176 feature columns
  Irpin: 176 feature columns
  Makariv: 128 feature columns
  Moschun: 176 feature columns
  Okhtyrka: 176 feature columns
  Trostianets: 128 feature columns
  Volnovakha: 176 feature columns
    impute_wide_parquet(rolling_stats): 409020 NaN -> 0 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll7_t1.json + bda_rolling_stats_roll7

# CELL 22: A18 -- bda_rolling_stats_roll13 (wide: rolling-stats sample at points, window=13)

In [22]:
# @title CELL 22: bda_rolling_stats_roll13_t{tier}.parquet

def _sample_dir_at_points(pts, ref_shape, tif_dir, prefix_filter=None):
    feats = {}
    if not tif_dir.exists():
        return feats
    for tif in sorted(tif_dir.glob("*.tif")):
        stem = tif.stem
        if prefix_filter and not any(stem.startswith(p) for p in prefix_filter):
            continue
        for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
            feats[_v4_col(stem, _v4_stat)] = _v4_arr
    return feats

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_stats_roll13', tier)
    if not FR_ROLLING_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_stats_roll13_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING STATS ROLL13 ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        feats = {}
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats",
            prefix_filter=["s1__vv__baseline", "s1__vh__baseline"]))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_stats",
            prefix_filter=["s1__vv__roll13__", "s1__vh__roll13__"]))
        feats.update(_sample_dir_at_points(pts, ref_shape,
            STACK_ROOT / city_name / "temporal" / "COH" / "rolling_stats",
            prefix_filter=["s1__coh_vv__roll13__"]))

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'rolling_stats')
        out = save_v5_parquet(df, 'rolling_stats_roll13', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_stats_roll13', 'A18', 'wide', ['city', 'point_id'], feat_cols,
                          'Q4: Rolling stats (window=13)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling stats roll13 for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING STATS ROLL13 ---
  Lysychansk: 176 feature columns
  Mariupol: 176 feature columns
  Rubizhne: 128 feature columns
  Sievierodonetsk: 176 feature columns
    impute_wide_parquet(rolling_stats): 518424 NaN -> 0 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll13_t0.json + bda_rolling_stats_roll13_t0_features.json (176 features)
  Saved: bda_rolling_stats_roll13_t0.parquet (39997 rows, 179 cols, 104s)

--- Tier 1 ROLLING STATS ROLL13 ---
  Borodyanka: 176 feature columns
  Bucha: 176 feature columns
  Chernihiv: 128 feature columns
  Dmytrivka: 176 feature columns
  Hostomel: 176 feature columns
  Irpin: 176 feature columns
  Makariv: 128 feature columns
  Moschun: 176 feature columns
  Okhtyrka: 176 feature columns
  Trostianets: 128 feature columns
  Volnovakha: 176 feature columns
    impute_wide_parquet(rolling_stats): 409020 NaN -> 0 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll13_t1.json + bda_rolling_stats

# CELL 23: A19 -- bda_card_drop (wide: CARD z-score drop accumulator)

In [23]:
# @title CELL 23: bda_card_drop_t{tier}.parquet
CARD_DROP_PRODUCTS = [
    's1__vv__z_running_min.tif',
    's1__vv__drop_count.tif',
    's1__vv__date_first_drop.tif',
    's1__vv__date_worst_drop.tif',
    's1__vv__max_z_drop.tif',
    's1__vv__scenes_observed.tif',
    's1__vv__lu_transition.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('card_drop', tier)
    if not FR_CARD_DROP and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  card_drop_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} CARD DROP ACCUMULATOR ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        drop_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "card_drop_accumulator"
        if not drop_dir.exists():
            print(f"  {city_name}: no card_drop_accumulator, skip")
            continue

        feats = {}
        for tif_name in CARD_DROP_PRODUCTS:
            tif_path = drop_dir / tif_name
            if tif_path.exists():
                for _v4_stat, _v4_arr in sample_one_kernel(tif_path, pts, ref_shape).items():
                    feats[_v4_col(tif_path.stem, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'carddrop')
        out = save_v5_parquet(df, 'card_drop', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('card_drop', 'A19', 'wide', ['city', 'point_id'], feat_cols,
                          'Q: Does cumulative CARD z-score drop carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No CARD drop data for tier {tier}")

    gc.collect()



--- Tier 0 CARD DROP ACCUMULATOR ---
  Lysychansk: 49 feature columns
  Mariupol: 49 feature columns
  Rubizhne: 49 feature columns
  Sievierodonetsk: 49 feature columns
    impute_wide_parquet(carddrop): 1004343 NaN -> 0 NaN, was_observed_carddrop added
    profile -> bda_card_drop_t0.json + bda_card_drop_t0_features.json (49 features)
  Saved: bda_card_drop_t0.parquet (39997 rows, 52 cols, 13s)

--- Tier 1 CARD DROP ACCUMULATOR ---
  Bucha: 49 feature columns
  Chernihiv: 49 feature columns
  Dmytrivka: 49 feature columns
  Hostomel: 49 feature columns
  Irpin: 49 feature columns
  Makariv: 49 feature columns
  Moschun: 49 feature columns
  Okhtyrka: 49 feature columns
  Trostianets: 49 feature columns
  Volnovakha: 49 feature columns
    impute_wide_parquet(carddrop): 602924 NaN -> 0 NaN, was_observed_carddrop added
    profile -> bda_card_drop_t1.json + bda_card_drop_t1_features.json (49 features)
  Saved: bda_card_drop_t1.parquet (18513 rows, 52 cols, 6s)

--- Tier 2 CARD DROP AC

# CELL 24: A20 -- bda_ms_change (wide: MS SWIR/NBR change accumulator)

In [24]:
# @title CELL 24: bda_ms_change_t{tier}.parquet
MS_CHANGE_PRODUCTS = [
    's2__swir_z_running_max.tif',
    's2__nbr_z_abs_running_max.tif',
    's2__swir_rise_count.tif',
    's2__nbr_anomaly_count.tif',
    's2__date_first_swir_rise.tif',
    's2__date_worst_swir_rise.tif',
    's2__scenes_observed.tif',
    's2__lu_transition.tif',
    's2__swir_baseline_mean.tif',
    's2__swir_baseline_std.tif',
    's2__nbr_baseline_mean.tif',
    's2__nbr_baseline_std.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('ms_change', tier)
    if not FR_MS_CHANGE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  ms_change_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} MS CHANGE ACCUMULATOR ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        acc_dir = STACK_ROOT / city_name / "temporal" / "MS" / "ms_change_accumulator"
        if not acc_dir.exists():
            print(f"  {city_name}: no ms_change_accumulator, skip")
            continue

        feats = {}
        for tif_name in MS_CHANGE_PRODUCTS:
            tif_path = acc_dir / tif_name
            if tif_path.exists():
                for _v4_stat, _v4_arr in sample_one_kernel(tif_path, pts, ref_shape).items():
                    feats[_v4_col(tif_path.stem, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'mschange')
        out = save_v5_parquet(df, 'ms_change', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('ms_change', 'A20', 'wide', ['city', 'point_id'], feat_cols,
                          'Q: Does cumulative MS SWIR brightness + NBR anomaly carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No MS change data for tier {tier}")

    gc.collect()



--- Tier 0 MS CHANGE ACCUMULATOR ---
  Lysychansk: 57 feature columns
  Mariupol: no ms_change_accumulator, skip
  Rubizhne: 41 feature columns
  Sievierodonetsk: 41 feature columns
    impute_wide_parquet(mschange): 990774 NaN -> 990774 NaN, was_observed_mschange added
    profile -> bda_ms_change_t0.json + bda_ms_change_t0_features.json (57 features)
  Saved: bda_ms_change_t0.parquet (17382 rows, 60 cols, 1s)

--- Tier 1 MS CHANGE ACCUMULATOR ---
  Borodyanka: no ms_change_accumulator, skip
  Bucha: 57 feature columns
  Chernihiv: 41 feature columns
  Dmytrivka: 65 feature columns
  Hostomel: 41 feature columns
  Irpin: 57 feature columns
  Makariv: 41 feature columns
  Moschun: 41 feature columns
  Okhtyrka: 41 feature columns
  Trostianets: 41 feature columns
  Volnovakha: 57 feature columns
    impute_wide_parquet(mschange): 1203309 NaN -> 981189 NaN, was_observed_mschange added
    profile -> bda_ms_change_t1.json + bda_ms_change_t1_features.json (65 features)
  Saved: bda_ms_ch

# CELL 25: A21 -- bda_ms_maha (wide: MS Mahalanobis accumulator)

In [25]:
# @title CELL 25: bda_ms_maha_t{tier}.parquet
MS_MAHA_PRODUCTS = [
    's2__mahalanobis_running_max.tif',
    's2__mahalanobis_exceedance_count.tif',
    's2__date_first_exceedance.tif',
    's2__date_worst_exceedance.tif',
    's2__scenes_observed.tif',
    's2__lu_transition.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('ms_maha', tier)
    if not FR_MS_MAHA and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  ms_maha_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} MS MAHALANOBIS ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        acc_dir = STACK_ROOT / city_name / "temporal" / "MS" / "ms_mahalanobis_accumulator"
        if not acc_dir.exists():
            print(f"  {city_name}: no ms_mahalanobis_accumulator, skip")
            continue

        feats = {}
        for tif_name in MS_MAHA_PRODUCTS:
            tif_path = acc_dir / tif_name
            if tif_path.exists():
                for _v4_stat, _v4_arr in sample_one_kernel(tif_path, pts, ref_shape).items():
                    feats[_v4_col(tif_path.stem, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'msmaha')
        out = save_v5_parquet(df, 'ms_maha', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('ms_maha', 'A21', 'wide', ['city', 'point_id'], feat_cols,
                          'Q: Does multi-band MS Mahalanobis distance carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No MS Mahalanobis data for tier {tier}")

    gc.collect()



--- Tier 0 MS MAHALANOBIS ---
  Lysychansk: 41 feature columns
  Mariupol: no ms_mahalanobis_accumulator, skip
  Rubizhne: 41 feature columns
  Sievierodonetsk: 41 feature columns
    impute_wide_parquet(msmaha): 339675 NaN -> 0 NaN, was_observed_msmaha added
    profile -> bda_ms_maha_t0.json + bda_ms_maha_t0_features.json (41 features)
  Saved: bda_ms_maha_t0.parquet (17382 rows, 44 cols, 5s)

--- Tier 1 MS MAHALANOBIS ---
  Borodyanka: no ms_mahalanobis_accumulator, skip
  Bucha: 41 feature columns
  Chernihiv: 41 feature columns
  Dmytrivka: 41 feature columns
  Hostomel: 41 feature columns
  Irpin: 41 feature columns
  Makariv: 41 feature columns
  Moschun: 41 feature columns
  Okhtyrka: 41 feature columns
  Trostianets: 41 feature columns
  Volnovakha: 41 feature columns
    impute_wide_parquet(msmaha): 318231 NaN -> 0 NaN, was_observed_msmaha added
    profile -> bda_ms_maha_t1.json + bda_ms_maha_t1_features.json (41 features)
  Saved: bda_ms_maha_t1.parquet (18513 rows, 44 col

# CELL 26: A22 -- bda_lu_change (wide: landuse change accumulator)

In [26]:
# @title CELL 26: bda_lu_change_t{tier}.parquet
LU_CHANGE_PRODUCTS = [
    's2__lu__date_first_loss.tif',
    's2__lu__date_persistent_loss.tif',
    's2__lu__loss_count.tif',
    's2__lu__loss_fraction.tif',
    's2__lu__final_class.tif',
    's2__lu__modal_post_class.tif',
    's2__lu__urban_retained.tif',
    's2__lu__scenes_observed.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('lu_change', tier)
    if not FR_LU_CHANGE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  lu_change_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} LANDUSE CHANGE ACCUMULATOR ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue

        acc_dir = STACK_ROOT / city_name / "temporal" / "LANDUSE" / "landuse_change_accumulator"
        if not acc_dir.exists():
            print(f"  {city_name}: no landuse_change_accumulator, skip")
            continue

        feats = {}
        for tif_name in LU_CHANGE_PRODUCTS:
            tif_path = acc_dir / tif_name
            if tif_path.exists():
                for _v4_stat, _v4_arr in sample_one_kernel(tif_path, pts, ref_shape).items():
                    feats[_v4_col(tif_path.stem, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'luchange')
        out = save_v5_parquet(df, 'lu_change', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('lu_change', 'A22', 'wide', ['city', 'point_id'], feat_cols,
                          'Q: Does persistent urban-to-other landuse loss carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No landuse change data for tier {tier}")

    gc.collect()



--- Tier 0 LANDUSE CHANGE ACCUMULATOR ---
  Lysychansk: 50 feature columns
  Mariupol: 50 feature columns
  Rubizhne: 50 feature columns
  Sievierodonetsk: 50 feature columns
    impute_wide_parquet(luchange): 460844 NaN -> 0 NaN, was_observed_luchange added
    profile -> bda_lu_change_t0.json + bda_lu_change_t0_features.json (50 features)
  Saved: bda_lu_change_t0.parquet (39997 rows, 53 cols, 20s)

--- Tier 1 LANDUSE CHANGE ACCUMULATOR ---
  Borodyanka: no landuse_change_accumulator, skip
  Bucha: 50 feature columns
  Chernihiv: 50 feature columns
  Dmytrivka: 50 feature columns
  Hostomel: 50 feature columns
  Irpin: 50 feature columns
  Makariv: 50 feature columns
  Moschun: 50 feature columns
  Okhtyrka: 50 feature columns
  Trostianets: 50 feature columns
  Volnovakha: 50 feature columns
    impute_wide_parquet(luchange): 422864 NaN -> 0 NaN, was_observed_luchange added
    profile -> bda_lu_change_t1.json + bda_lu_change_t1_features.json (50 features)
  Saved: bda_lu_change_t1

# CELL 27: A23 -- bda_rolling_accum_coh (long: rolling-window matched-filter, COH)

In [27]:
# @title CELL 27: bda_rolling_accum_coh_t{tier}.parquet
ROLLING_ACCUM_OPS = ['running_min', 'running_max', 'max_abs_delta']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_accum_coh', tier)
    if not FR_ROLLING_ACCUM_COH and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_accum_coh_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING ACCUM COH ---")
    all_rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        roll_dir = STACK_ROOT / city_name / "temporal" / "COH" / "rolling_accum"
        if not roll_dir.exists():
            continue

        per_date_feats = {}
        for tif in sorted(roll_dir.glob("s1__coh_vv__roll*__*__*.tif")):
            m = re.match(r"s1__coh_vv__roll(\d+)__([a-z_]+)__(\d{8})\.tif$", tif.name)
            if not m:
                continue
            ws, op, date_str = m.group(1), m.group(2), m.group(3)
            if op not in ROLLING_ACCUM_OPS:
                continue
            col = f"s1__coh_vv__roll{ws}__{op}"
            for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                per_date_feats.setdefault(date_str, {})[_v4_col(col, _v4_stat)] = _v4_arr
        if not per_date_feats:
            continue

        for date_str, feats in per_date_feats.items():
            df_d = pd.DataFrame(feats)
            df_d['point_id'] = pts['point_id'].tolist()
            df_d['city'] = city_name
            df_d['date'] = date_str
            all_rows.append(df_d)

        print(f"  {city_name}: {len(per_date_feats)} dates")

    if all_rows:
        df = pd.concat(all_rows, ignore_index=True)
        out = save_v5_parquet(df, 'rolling_accum_coh', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_accum_coh', 'A23', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q4b: Rolling-window matched-filter signal (COH)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling_accum_coh data for tier {tier}")
    gc.collect()



--- Tier 0 ROLLING ACCUM COH ---
  Lysychansk: 3 dates
  Mariupol: 5 dates
  Rubizhne: 3 dates
  Sievierodonetsk: 4 dates
    profile -> bda_rolling_accum_coh_t0.json + bda_rolling_accum_coh_t0_features.json (48 features)
  Saved: bda_rolling_accum_coh_t0.parquet (169639 rows, 51 cols, 69s)

--- Tier 1 ROLLING ACCUM COH ---
  Borodyanka: 7 dates
  Bucha: 6 dates
  Dmytrivka: 5 dates
  Hostomel: 7 dates
  Irpin: 7 dates
  Moschun: 7 dates
  Okhtyrka: 7 dates
  Volnovakha: 5 dates
    profile -> bda_rolling_accum_coh_t1.json + bda_rolling_accum_coh_t1_features.json (48 features)
  Saved: bda_rolling_accum_coh_t1.parquet (97703 rows, 51 cols, 53s)

--- Tier 2 ROLLING ACCUM COH ---
  Avdiivka: 61 dates
  Chornobaivka: 8 dates
  Kherson: 8 dates
  Kramatorsk: 50 dates
    profile -> bda_rolling_accum_coh_t2.json + bda_rolling_accum_coh_t2_features.json (72 features)
  Saved: bda_rolling_accum_coh_t2.parquet (83618 rows, 75 cols, 146s)


# CELL 28: A24 -- bda_rolling_accum_card (long: rolling-window matched-filter, CARD)

In [28]:
# @title CELL 28: bda_rolling_accum_card_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_accum_card', tier)
    if not FR_ROLLING_ACCUM_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_accum_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING ACCUM CARD ---")
    all_rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        roll_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_accum"
        if not roll_dir.exists():
            continue

        per_date_feats = {}
        for tif in sorted(roll_dir.glob("s1__*__roll*__*__*.tif")):
            m = re.match(r"s1__(vv|vh)__roll(\d+)__([a-z_]+)__(\d{8})\.tif$", tif.name)
            if not m:
                continue
            pol, ws, op, date_str = m.group(1), m.group(2), m.group(3), m.group(4)
            if op not in ROLLING_ACCUM_OPS:
                continue
            col = f"s1__{pol}__roll{ws}__{op}"
            for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                per_date_feats.setdefault(date_str, {})[_v4_col(col, _v4_stat)] = _v4_arr
        if not per_date_feats:
            continue

        for date_str, feats in per_date_feats.items():
            df_d = pd.DataFrame(feats)
            df_d['point_id'] = pts['point_id'].tolist()
            df_d['city'] = city_name
            df_d['date'] = date_str
            all_rows.append(df_d)
        print(f"  {city_name}: {len(per_date_feats)} dates")

    if all_rows:
        df = pd.concat(all_rows, ignore_index=True)
        out = save_v5_parquet(df, 'rolling_accum_card', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_accum_card', 'A24', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q4b: Rolling-window matched-filter signal (CARD)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling_accum_card data for tier {tier}")
    gc.collect()



--- Tier 0 ROLLING ACCUM CARD ---
  Lysychansk: 6 dates
  Mariupol: 12 dates
  Rubizhne: 11 dates
  Sievierodonetsk: 10 dates
    profile -> bda_rolling_accum_card_t0.json + bda_rolling_accum_card_t0_features.json (144 features)
  Saved: bda_rolling_accum_card_t0.parquet (418354 rows, 147 cols, 581s)

--- Tier 1 ROLLING ACCUM CARD ---
  Borodyanka: 8 dates
  Bucha: 7 dates
  Chernihiv: 8 dates
  Dmytrivka: 7 dates
  Hostomel: 8 dates
  Irpin: 8 dates
  Makariv: 7 dates
  Moschun: 8 dates
  Okhtyrka: 8 dates
  Trostianets: 7 dates
  Volnovakha: 6 dates
    profile -> bda_rolling_accum_card_t1.json + bda_rolling_accum_card_t1_features.json (96 features)
  Saved: bda_rolling_accum_card_t1.parquet (143075 rows, 99 cols, 205s)

--- Tier 2 ROLLING ACCUM CARD ---
  Avdiivka: 63 dates
  Chornobaivka: 25 dates
  Kharkiv: 22 dates
  Kherson: 26 dates
  Kramatorsk: 123 dates
  Mykolaiv: 1 dates
    profile -> bda_rolling_accum_card_t2.json + bda_rolling_accum_card_t2_features.json (144 features)

# CELL 29: A25 -- bda_rolling_accum_ms (long: rolling-window matched-filter, MS)

In [29]:
# @title CELL 29: bda_rolling_accum_ms_t{tier}.parquet
MS_ROLLING_BANDS = ['b11', 'b12', 'b08', 'b8a', 'nbr']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('rolling_accum_ms', tier)
    if not FR_ROLLING_ACCUM_MS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_accum_ms_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING ACCUM MS ---")
    all_rows = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        roll_dir = STACK_ROOT / city_name / "temporal" / "MS" / "rolling_accum"
        if not roll_dir.exists():
            continue

        per_date_feats = {}
        for tif in sorted(roll_dir.glob("s2__*__roll*__*__*.tif")):
            m = re.match(r"s2__([a-z0-9]+)__roll(\d+)__([a-z_]+)__(\d{8})\.tif$", tif.name)
            if not m:
                continue
            band, ws, op, date_str = m.group(1), m.group(2), m.group(3), m.group(4)
            if band not in MS_ROLLING_BANDS:
                continue
            if op not in ROLLING_ACCUM_OPS:
                continue
            col = f"s2__{band}__roll{ws}__{op}"
            for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                per_date_feats.setdefault(date_str, {})[_v4_col(col, _v4_stat)] = _v4_arr
        if not per_date_feats:
            continue

        for date_str, feats in per_date_feats.items():
            df_d = pd.DataFrame(feats)
            df_d['point_id'] = pts['point_id'].tolist()
            df_d['city'] = city_name
            df_d['date'] = date_str
            all_rows.append(df_d)
        print(f"  {city_name}: {len(per_date_feats)} dates")

    if all_rows:
        df = pd.concat(all_rows, ignore_index=True)
        out = save_v5_parquet(df, 'rolling_accum_ms', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_accum_ms', 'A25', 'long', ['city', 'point_id', 'date'], feat_cols,
                          'Q4b: Rolling-window matched-filter signal (MS)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling_accum_ms data for tier {tier}")
    gc.collect()



--- Tier 0 ROLLING ACCUM MS ---
  Lysychansk: 12 dates
  Mariupol: 10 dates
  Rubizhne: 11 dates
  Sievierodonetsk: 13 dates
    profile -> bda_rolling_accum_ms_t0.json + bda_rolling_accum_ms_t0_features.json (360 features)
  Saved: bda_rolling_accum_ms_t0.parquet (434150 rows, 363 cols, 174s)

--- Tier 1 ROLLING ACCUM MS ---
  Bucha: 11 dates
  Chernihiv: 9 dates
  Dmytrivka: 9 dates
  Hostomel: 8 dates
  Irpin: 7 dates
  Makariv: 9 dates
  Moschun: 8 dates
  Okhtyrka: 12 dates
  Trostianets: 9 dates
  Volnovakha: 10 dates
    profile -> bda_rolling_accum_ms_t1.json + bda_rolling_accum_ms_t1_features.json (360 features)
  Saved: bda_rolling_accum_ms_t1.parquet (161606 rows, 363 cols, 138s)

--- Tier 2 ROLLING ACCUM MS ---
  Avdiivka: 26 dates
  Chornobaivka: 17 dates
  Kharkiv: 13 dates
  Kherson: 17 dates
  Kramatorsk: 79 dates
    profile -> bda_rolling_accum_ms_t2.json + bda_rolling_accum_ms_t2_features.json (360 features)
  Saved: bda_rolling_accum_ms_t2.parquet (74363 rows, 363 

# CELL 30: A26 -- bda_block_accum_coh (wide: block-scope matched-filter, COH)

In [30]:
# @title CELL 30: bda_block_accum_coh_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('block_accum_coh', tier)
    if not FR_BLOCK_ACCUM_COH and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  block_accum_coh_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK ACCUM COH ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        blk_dir = STACK_ROOT / city_name / "temporal" / "COH" / "block_accum"
        if not blk_dir.exists():
            continue

        feats = {}
        for tif in sorted(blk_dir.glob("s1__coh_vv__*__*.tif")):
            m = re.match(r"s1__coh_vv__(blk\w+)__([a-z_]+)\.tif$", tif.name)
            if not m:
                continue
            block, op = m.group(1), m.group(2)
            col = f"s1__coh_vv__{block}__{op}"
            for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                feats[_v4_col(col, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'block_accum_coh')
        out = save_v5_parquet(df, 'block_accum_coh', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('block_accum_coh', 'A26', 'wide', ['city', 'point_id'], feat_cols,
                          'Q6b: Block-scope matched-filter signal (COH)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block_accum_coh data for tier {tier}")
    gc.collect()



--- Tier 0 BLOCK ACCUM COH ---
  Lysychansk: 32 feature columns
  Mariupol: 32 feature columns
  Rubizhne: 16 feature columns
  Sievierodonetsk: 16 feature columns
    impute_wide_parquet(block_accum_coh): 779596 NaN -> 0 NaN, was_observed_block_accum_coh added
    profile -> bda_block_accum_coh_t0.json + bda_block_accum_coh_t0_features.json (32 features)
  Saved: bda_block_accum_coh_t0.parquet (39997 rows, 35 cols, 6s)

--- Tier 1 BLOCK ACCUM COH ---
  Borodyanka: 32 feature columns
  Bucha: 32 feature columns
  Chernihiv: 16 feature columns
  Dmytrivka: 32 feature columns
  Hostomel: 32 feature columns
  Irpin: 32 feature columns
  Moschun: 32 feature columns
  Okhtyrka: 32 feature columns
  Volnovakha: 32 feature columns
    impute_wide_parquet(block_accum_coh): 288326 NaN -> 0 NaN, was_observed_block_accum_coh added
    profile -> bda_block_accum_coh_t1.json + bda_block_accum_coh_t1_features.json (32 features)
  Saved: bda_block_accum_coh_t1.parquet (18429 rows, 35 cols, 6s)

--- 

# CELL 31: A27 -- bda_block_accum_card (wide: block-scope matched-filter, CARD)

In [31]:
# @title CELL 31: bda_block_accum_card_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('block_accum_card', tier)
    if not FR_BLOCK_ACCUM_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  block_accum_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK ACCUM CARD ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        blk_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "block_accum"
        if not blk_dir.exists():
            continue

        feats = {}
        for tif in sorted(blk_dir.glob("s1__*__*__*.tif")):
            m = re.match(r"s1__(vv|vh)__(blk\w+)__([a-z_]+)\.tif$", tif.name)
            if not m:
                continue
            pol, block, op = m.group(1), m.group(2), m.group(3)
            col = f"s1__{pol}__{block}__{op}"
            for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                feats[_v4_col(col, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'block_accum_card')
        out = save_v5_parquet(df, 'block_accum_card', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('block_accum_card', 'A27', 'wide', ['city', 'point_id'], feat_cols,
                          'Q6b: Block-scope matched-filter signal (CARD)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block_accum_card data for tier {tier}")
    gc.collect()



--- Tier 0 BLOCK ACCUM CARD ---
  Lysychansk: 80 feature columns
  Mariupol: 120 feature columns
  Rubizhne: 80 feature columns
  Sievierodonetsk: 80 feature columns
    impute_wide_parquet(block_accum_card): 2448874 NaN -> 1599880 NaN, was_observed_block_accum_card added
    profile -> bda_block_accum_card_t0.json + bda_block_accum_card_t0_features.json (120 features)
  Saved: bda_block_accum_card_t0.parquet (39997 rows, 123 cols, 34s)

--- Tier 1 BLOCK ACCUM CARD ---
  Borodyanka: 80 feature columns
  Bucha: 80 feature columns
  Chernihiv: 80 feature columns
  Dmytrivka: 80 feature columns
  Hostomel: 80 feature columns
  Irpin: 80 feature columns
  Makariv: 80 feature columns
  Moschun: 80 feature columns
  Okhtyrka: 80 feature columns
  Trostianets: 80 feature columns
  Volnovakha: 80 feature columns
    impute_wide_parquet(block_accum_card): 608852 NaN -> 456480 NaN, was_observed_block_accum_card added
    profile -> bda_block_accum_card_t1.json + bda_block_accum_card_t1_features

# CELL 32: A28 -- bda_block_accum_ms (wide: block-scope matched-filter, MS)

In [32]:
# @title CELL 32: bda_block_accum_ms_t{tier}.parquet
MS_BLOCK_BANDS = ['b11', 'b12', 'b08', 'b8a', 'nbr']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v5_path('block_accum_ms', tier)
    if not FR_BLOCK_ACCUM_MS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  block_accum_ms_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK ACCUM MS ---")
    all_city_dfs = []

    for city_name in tier_cities:
        pts, ref_shape, _ = build_or_load_points(city_name)
        if pts is None or len(pts) == 0:
            continue
        blk_dir = STACK_ROOT / city_name / "temporal" / "MS" / "block_accum"
        if not blk_dir.exists():
            continue

        feats = {}
        for tif in sorted(blk_dir.glob("s2__*__*__*.tif")):
            m = re.match(r"s2__([a-z0-9]+)__(blk\w+)__([a-z_]+)\.tif$", tif.name)
            if not m:
                continue
            band, block, op = m.group(1), m.group(2), m.group(3)
            if band not in MS_BLOCK_BANDS:
                continue
            col = f"s2__{band}__{block}__{op}"
            for _v4_stat, _v4_arr in sample_one_kernel(tif, pts, ref_shape).items():
                feats[_v4_col(col, _v4_stat)] = _v4_arr
        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['point_id'] = pts['point_id'].tolist()
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        df = impute_wide_parquet(df, 'block_accum_ms')
        out = save_v5_parquet(df, 'block_accum_ms', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('block_accum_ms', 'A28', 'wide', ['city', 'point_id'], feat_cols,
                          'Q6b: Block-scope matched-filter signal (MS)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block_accum_ms data for tier {tier}")
    gc.collect()



--- Tier 0 BLOCK ACCUM MS ---
  Lysychansk: 840 feature columns
  Mariupol: 840 feature columns
  Rubizhne: 1176 feature columns
  Sievierodonetsk: 1176 feature columns
    impute_wide_parquet(block_accum_ms): 52504096 NaN -> 17038722 NaN, was_observed_block_accum_ms added
    profile -> bda_block_accum_ms_t0.json + bda_block_accum_ms_t0_features.json (1344 features)
  Saved: bda_block_accum_ms_t0.parquet (39997 rows, 1347 cols, 41s)

--- Tier 1 BLOCK ACCUM MS ---
  Bucha: 504 feature columns
  Chernihiv: 840 feature columns
  Dmytrivka: 504 feature columns
  Hostomel: 672 feature columns
  Irpin: 504 feature columns
  Makariv: 504 feature columns
  Moschun: 672 feature columns
  Okhtyrka: 672 feature columns
  Trostianets: 672 feature columns
  Volnovakha: 840 feature columns
    impute_wide_parquet(block_accum_ms): 18102128 NaN -> 4628250 NaN, was_observed_block_accum_ms added
    profile -> bda_block_accum_ms_t1.json + bda_block_accum_ms_t1_features.json (1008 features)
  Saved: bd

# CELL 33: FUSION PARQUETS (F1-F8)

Pre-joined fusions for ML experiments. All joins use `city + point_id [+ date]` as keys.

| ID | Name | Composition | Join shape |
|---|---|---|---|
| F1 | fusion_ms_card | scene_ms + scene_card | long+long (date-aligned) |
| F2 | fusion_ms_card_cohdrop | scene_ms + scene_card + coh_drop | long+long+wide |
| F3 | fusion_card_cohdrop | scene_card + coh_drop | long+wide |
| F4 | fusion_ms_cohdrop | scene_ms + coh_drop | long+wide |
| F5 | fusion_indices_card | scene_indices + scene_card | long+long |
| F6 | fusion_indices_card_cohdrop | scene_indices + scene_card + coh_drop | long+long+wide |
| F7 | fusion_composite_cohdrop | composite_prepost_bands + coh_drop | wide+wide |
| F8 | fusion_composite_blockstats | composite_prepost_bands + block_stats | wide+wide |


In [33]:
# @title CELL 33: FUSION PARQUETS (F1-F8)
def load_v3_pq(name, tier):
    p = v5_path(name, tier)
    if not p.exists():
        return None
    return pd.read_parquet(p)

def join_long_long(df_a, df_b, name_a, name_b):
    join_cols = ['city', 'point_id', 'date']
    meta_a = [c for c in df_a.columns if _is_metadata_column(c)]
    meta_b = [c for c in df_b.columns if _is_metadata_column(c)]
    feat_a = [c for c in df_a.columns if c not in meta_a]
    feat_b = [c for c in df_b.columns if c not in meta_b]
    df = df_a.merge(
        df_b.drop(columns=[c for c in meta_b if c in meta_a and c not in join_cols], errors='ignore'),
        on=join_cols, how='outer',
    )
    df[f'was_observed_{name_a}'] = df[feat_a[0]].notna().astype(int) if feat_a else 1
    df[f'was_observed_{name_b}'] = df[feat_b[0]].notna().astype(int) if feat_b else 1
    return df

def join_long_wide(df_long, df_wide, wide_name):
    join_cols = ['city', 'point_id']
    meta_wide = [c for c in df_wide.columns if _is_metadata_column(c)]
    feat_wide = [c for c in df_wide.columns if c not in meta_wide]
    df = df_long.merge(df_wide[join_cols + feat_wide], on=join_cols, how='left')
    df[f'was_observed_{wide_name}'] = df[feat_wide[0]].notna().astype(int) if feat_wide else 0
    return df

def join_wide_wide(df_a, df_b, name_a, name_b):
    join_cols = ['city', 'point_id']
    meta_a = [c for c in df_a.columns if _is_metadata_column(c)]
    meta_b = [c for c in df_b.columns if _is_metadata_column(c)]
    feat_a = [c for c in df_a.columns if c not in meta_a]
    feat_b = [c for c in df_b.columns if c not in meta_b]
    df = df_a.merge(df_b[join_cols + feat_b], on=join_cols, how='outer')
    df[f'was_observed_{name_a}'] = df[feat_a[0]].notna().astype(int) if feat_a else 1
    df[f'was_observed_{name_b}'] = df[feat_b[0]].notna().astype(int) if feat_b else 1
    return df

FUSIONS = [
    ('fusion_ms_card',              'F1', 'long', ['scene_ms', 'scene_card'],                          'Q2: Does MS+CARD beat either alone?'),
    ('fusion_ms_card_cohdrop',      'F2', 'long', ['scene_ms', 'scene_card', 'coh_drop'],              'Q2: Full multimodal'),
    ('fusion_card_cohdrop',         'F3', 'long', ['scene_card', 'coh_drop'],                          'Q2: SAR-only multimodal'),
    ('fusion_ms_cohdrop',           'F4', 'long', ['scene_ms', 'coh_drop'],                            'Q2: Optical + COH drop'),
    ('fusion_indices_card',         'F5', 'long', ['scene_indices', 'scene_card'],                     'Q2: Indices + CARD'),
    ('fusion_indices_card_cohdrop', 'F6', 'long', ['scene_indices', 'scene_card', 'coh_drop'],         'Q2: Indices + CARD + COH drop'),
    ('fusion_composite_cohdrop',    'F7', 'wide', ['composite_prepost_bands', 'coh_drop'],             'Q2: Composite + COH drop'),
    ('fusion_composite_blockstats', 'F8', 'wide', ['composite_prepost_bands', 'block_stats'],          'Q6: Dietrich composite + block replication'),
]

for tier, tier_cities in TIER_CITIES.items():
    for fusion_name, fid, fmt, sources, question in FUSIONS:
        out_path = v5_path(fusion_name, tier)
        if not FR_FUSIONS and out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  {fusion_name}_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

        t0 = time.time()
        print(f"\n--- {fusion_name} tier {tier} ---")

        dfs = {}
        missing = False
        for src in sources:
            df_src = load_v3_pq(src, tier)
            if df_src is None:
                print(f"  SKIP: {src}_t{tier} not found")
                missing = True
                break
            dfs[src] = df_src

        if missing:
            continue

        # inner-join on shared cities (avoid structural NaN from missing modalities)
        city_sets = {src: set(dfs[src]['city'].unique()) for src in sources}
        shared_cities = set.intersection(*city_sets.values())
        excluded = sorted(set.union(*city_sets.values()) - shared_cities)
        if excluded:
            for src in sources:
                dfs[src] = dfs[src][dfs[src]['city'].isin(shared_cities)].reset_index(drop=True)
            print(f"  inner-join on {len(shared_cities)} shared cities (excluded: {excluded})")

        is_long = {src: 'date' in dfs[src].columns for src in sources}

        result = None
        for i, src in enumerate(sources):
            if i == 0:
                result = dfs[src]
                continue
            if is_long.get(sources[0]) and is_long.get(src):
                result = join_long_long(result, dfs[src], sources[0], src)
            elif is_long.get(sources[0]) and not is_long.get(src):
                result = join_long_wide(result, dfs[src], src)
            elif not is_long.get(sources[0]) and not is_long.get(src):
                result = join_wide_wide(result, dfs[src], sources[0], src)
            else:
                result = join_long_wide(dfs[src], result, sources[0])

        if result is not None and len(result) > 0:
            out = save_v5_parquet(result, fusion_name, tier)
            feat_cols = [c for c in result.columns if not _is_metadata_column(c)]
            join_keys = ['city', 'point_id'] + (['date'] if 'date' in result.columns else [])
            register_manifest(fusion_name, fid, fmt, join_keys, feat_cols, question,
                              tier, len(result), composed_of=sources,
                              cities_excluded=excluded if excluded else None)
            print(f"  Saved: {out.name} ({len(result)} rows, {len(result.columns)} cols, {time.time()-t0:.0f}s)")
        else:
            print(f"  Empty fusion for {fusion_name} tier {tier}")

        gc.collect()



--- fusion_ms_card tier 0 ---
    profile -> bda_fusion_ms_card_t0.json + bda_fusion_ms_card_t0_features.json (112 features)
  Saved: bda_fusion_ms_card_t0.parquet (944422 rows, 119 cols, 10s)

--- fusion_ms_card_cohdrop tier 0 ---
  inner-join on 3 shared cities (excluded: ['Rubizhne'])
    profile -> bda_fusion_ms_card_cohdrop_t0.json + bda_fusion_ms_card_cohdrop_t0_features.json (161 features)
  Saved: bda_fusion_ms_card_cohdrop_t0.parquet (829376 rows, 169 cols, 12s)

--- fusion_card_cohdrop tier 0 ---
  inner-join on 3 shared cities (excluded: ['Rubizhne'])
    profile -> bda_fusion_card_cohdrop_t0.json + bda_fusion_card_cohdrop_t0_features.json (65 features)
  Saved: bda_fusion_card_cohdrop_t0.parquet (433322 rows, 71 cols, 3s)

--- fusion_ms_cohdrop tier 0 ---
  inner-join on 3 shared cities (excluded: ['Rubizhne'])
    profile -> bda_fusion_ms_cohdrop_t0.json + bda_fusion_ms_cohdrop_t0_features.json (145 features)
  Saved: bda_fusion_ms_cohdrop_t0.parquet (411978 rows, 151 col

# CELL 34: GROUPKFOLD ASSIGNMENT (per-city fold tags for NB06+ ML splits)

In [34]:
# @title CELL 34: GROUPKFOLD
import stack_groupkfold
importlib.reload(stack_groupkfold)
from stack_groupkfold import run as run_groupkfold

v5_gkf_path = V5_DIR / "groupkfold.parquet"
if FR_GROUPKFOLD or not v5_gkf_path.exists():
    GKF = run_groupkfold(stack_root=STACK_ROOT, n_folds=None, output_path=v5_gkf_path)
else:
    print(f"  GroupKFold exists, skipping")


GROUPKFOLD: CROSS-VALIDATION FOLD ASSIGNMENT BY CITY

  Total cities: 21
  ML-ready cities: 21
  Fold mode: leave_one_city_out
  N folds: 21

  City                   Fold    Bldg    Dmg  CARD  COH   MS
  ---------------------- ----  ------  -----  ----  ----  ----
  Kharkiv                  0   213845    238    Y     -     Y
  Mykolaiv                 1   137326     84    Y     Y     -
  Mariupol                 2   119107   3055    Y     Y     Y
  Chernihiv                3    58626    333    Y     Y     Y
  Kramatorsk               4    58518     19    Y     Y     Y
  Borodyanka               5    49969     62    Y     Y     -
  Lysychansk               6    47169   1071    Y     Y     Y
  Okhtyrka                 7    27603     35    Y     Y     Y
  Moschun                  8    20565     73    Y     Y     Y
  Hostomel                 9    19798    511    Y     Y     Y
  Trostianets             10    18656     18    Y     -     Y
  Irpin                   11    17686    249    Y   

# CELL 35: WRITE parquet_manifest.json

In [35]:
# @title CELL 35: WRITE parquet_manifest.json
from datetime import datetime as _dt

# back-fill MANIFEST_ENTRIES from on-disk parquets that were skipped this run
scan_disk_and_register()

manifest_out = {
    'version': 'v3',
    'created': _dt.now().isoformat(),
    'created_by': 'NB05bV5 v1',
    'tier_selection': TIER_SELECTION,
    'sample_unit': 'point',
    'neg_ratio': NEG_RATIO,
    'neg_min_dist_m': NEG_MIN_DIST_M,
    'drop_excluded': DROP_EXCLUDED,
    'nan_handling': 'global_median_impute + was_observed flags for residual NaN',
    'parquets': MANIFEST_ENTRIES,
}

manifest_path = V5_DIR / 'parquet_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest_out, f, indent=2, default=str)

print(f"Manifest written: {manifest_path}")
print(f"  {len(MANIFEST_ENTRIES)} parquet definitions")
for name, info in sorted(MANIFEST_ENTRIES.items()):
    tiers = info.get('tiers_built', [])
    n_feat = info.get('n_features', 0)
    print(f"  {name:<35s} [{info['id']}] {info['format']:>5s} {n_feat:>4d} features, tiers={tiers}")


  disk-scan summary: +0 added, 108 already-registered, 0 unknown-name
Manifest written: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/v5/parquet_manifest.json
  36 parquet definitions
  block_accum_card                    [A27]  wide  680 features, tiers=[0, 1, 2]
  block_accum_coh                     [A26]  wide  144 features, tiers=[0, 1, 2]
  block_accum_ms                      [A28]  wide 3192 features, tiers=[0, 1, 2]
  block_stats                         [A15]  wide 2400 features, tiers=[0, 1, 2]
  card_drop                           [A19]  wide   49 features, tiers=[0, 1, 2]
  coh_drop                            [A14]  wide   49 features, tiers=[0, 1, 2]
  composite_prepost_bands             [A9]  wide  504 features, tiers=[0, 1, 2]
  composite_prepost_landuse           [A10]  wide    4 features, tiers=[0, 1, 2]
  composite_vs_scenes_landuse         [A12]  long    3 features, tiers=[0, 1, 2]
  fusion_card_cohdrop                 [F3]  long   65 features, tiers=[0, 1, 2]
  fusi

# CELL 36: VALIDATE V3 PARQUETS

In [36]:
# @title CELL 36: VALIDATE V3 PARQUETS
print("=" * 70)
print("VALIDATE: CHECK ALL V3 PER-TIER PARQUETS")
print("=" * 70)

parquet_files = sorted(V5_DIR.glob("bda_*_t*.parquet"))
print(f"\n  Per-tier parquets found: {len(parquet_files)}")

for pq in parquet_files:
    df = pd.read_parquet(pq)
    n_cities = df['city'].nunique() if 'city' in df.columns else 0
    n_rows = len(df)
    n_cols = len(df.columns)
    size_mb = pq.stat().st_size / 1e6

    warn = ""
    if 'date' not in df.columns and 'date1' not in df.columns:
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        struct_nan = 0
        for col in feat_cols:
            nan_cities = set()
            ok_cities = set()
            for city, grp in df.groupby('city'):
                if grp[col].isna().all():
                    nan_cities.add(city)
                elif grp[col].notna().any():
                    ok_cities.add(city)
            if nan_cities and ok_cities:
                struct_nan += 1
        if struct_nan > 0:
            warn = f" WARNING: {struct_nan} structural NaN cols!"

    print(f"  {pq.name:<55s} {n_rows:>8d} rows  {n_cols:>4d} cols  {n_cities:>2d} cities  {size_mb:>6.1f} MB{warn}")

manifest_path = V5_DIR / 'parquet_manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        mf = json.load(f)
    print(f"\n  Manifest: {len(mf['parquets'])} entries")
    for name, info in mf['parquets'].items():
        for tier in TIER_SELECTION:
            pq_path = V5_DIR / f"bda_{name}_t{tier}.parquet"
            status = "OK" if pq_path.exists() else "MISSING"
            if status == "MISSING":
                print(f"  WARNING: {pq_path.name} in manifest but {status}")
else:
    print("  WARNING: parquet_manifest.json not found!")

print(f"\n{'='*70}")
print("VALIDATE COMPLETE")
print(f"{'='*70}")


VALIDATE: CHECK ALL V3 PER-TIER PARQUETS

  Per-tier parquets found: 108
  bda_block_accum_card_t0.parquet                            39997 rows   123 cols   4 cities     8.2 MB
  bda_block_accum_card_t1.parquet                            19020 rows    83 cols  11 cities     3.8 MB
  bda_block_accum_card_t2.parquet                             4226 rows   683 cols   6 cities     3.0 MB
  bda_block_accum_coh_t0.parquet                             39997 rows    35 cols   4 cities     1.8 MB
  bda_block_accum_coh_t1.parquet                             18429 rows    35 cols   9 cities     1.1 MB
  bda_block_accum_coh_t2.parquet                              1650 rows   147 cols   4 cities     0.7 MB
  bda_block_accum_ms_t0.parquet                              39997 rows  1347 cols   4 cities    10.0 MB
  bda_block_accum_ms_t1.parquet                              18513 rows  1011 cols  10 cities     3.8 MB
  bda_block_accum_ms_t2.parquet                               3533 rows  3195 cols   5 